In [1]:
# Imports (todos nesta célula)
import warnings

# O category_encoders (CountEncoder e TargetEncoder, usados no 2.3) vem do environment.yml
# (ver o README)
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import (MinMaxScaler, OneHotEncoder, OrdinalEncoder, RobustScaler,
                                   StandardScaler)
from category_encoders import CountEncoder, TargetEncoder   # W4, passo 1

# Como nas práticas (W5 a W9), o que foi escolhido está num ficheiro que todos os notebooks
# importam: notebooks/preprocessing.py. As constantes vêm de lá:
#   TARGET, GROUP (customer_id), DATE (purchase_date), RANDOM_STATE = 42,
#   OUT_OF_MODEL = [GROUP, DATE] (ficam na tabela, mas nunca entram no modelo) e
#   METRIC = "f1" (escolhida pela equipa na tarefa 3; a accuracy aparece ao lado)
from preprocessing import DATE, GROUP, METRIC, OUT_OF_MODEL, RANDOM_STATE, TARGET

# Como na W3 (células 5 e 6): sem scaler, a regressão logística não converge em 1000
# iterações (2.4). A causa é conhecida (escalas diferentes), a solução é escalar, e o
# efeito foi medido: o braço sem scaler fica de fora. Por isso o aviso é silenciado.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
# ===== [2.0] BASE COMUM - início =====
# Passo 1 - dados de partida, ordenados por data como na W5 (secção 3.4).
# As linhas sem data não têm lugar na cronologia, por isso ficam de fora.
# load_train (preprocessing.py) lê o train.csv, tira as linhas sem data e ordena por data com
# sort_values(kind="stable"), que mantém pela ordem do ficheiro as compras do mesmo dia (W5)
from preprocessing import load_train

raw, frame = load_train("../data/train.csv")

print("linhas excluídas por não terem data:", len(raw) - len(frame))
print(f"linhas na cronologia: {len(frame):,}, de {frame[DATE].iloc[0]} a {frame[DATE].iloc[-1]}")

linhas excluídas por não terem data: 65
linhas na cronologia: 6,469, de 2015-07-11 a 2025-02-15


In [3]:
# Passo 2 - que pergunta faz o test.csv? Só a estrutura (datas e clientes); nenhum valor
# do teste é usado para ajustar nada. As datas comparam-se como texto, como na EDA.
test_csv = pd.read_csv("../data/test.csv", index_col="ID")
test_rows_seen = test_csv[GROUP].isin(raw[GROUP])

pd.Series({
    "datas do treino": f"{frame[DATE].iloc[0]} a {frame[DATE].iloc[-1]}",
    "datas do teste": f"{test_csv[DATE].min()} a {test_csv[DATE].max()}",
    "clientes do teste já vistos no treino":
        f"{pd.Series(test_csv[GROUP].unique()).isin(raw[GROUP]).sum()} de {test_csv[GROUP].nunique()}",
    "linhas do teste com cliente já visto":
        f"{test_rows_seen.sum()} de {len(test_csv)} ({100 * test_rows_seen.mean():.1f}%)",
}).to_frame("o que o teste pergunta")

,o que o teste pergunta
datas do treino,2015-07-11 a 2025-02-15
datas do teste,2025-05-24 a 2026-10-06
clientes do teste já vistos no treino,184 de 206
linhas do teste com cliente já visto,932 de 986 (94.5%)


In [4]:
# Passo 3 - divisões por tempo: cada divisão treina no passado e valida no que vem a seguir,
# como o test.csv, que tem compras posteriores às do train.csv
# time_splits (preprocessing.py): list(TimeSeriesSplit(n_splits=5).split(frame))
from preprocessing import time_splits

splits = time_splits(frame)

for fold, (train_idx, val_idx) in enumerate(splits, start=1):
    print(f"divisão {fold}: treino até {frame[DATE].iloc[train_idx].max()}"
          f" ({len(train_idx):,} linhas), valida de {frame[DATE].iloc[val_idx].min()}"
          f" a {frame[DATE].iloc[val_idx].max()} ({len(val_idx):,} linhas)")

divisão 1: treino até 2017-08-12 (1,079 linhas), valida de 2017-08-12 a 2019-01-12 (1,078 linhas)
divisão 2: treino até 2019-01-12 (2,157 linhas), valida de 2019-01-12 a 2020-10-03 (1,078 linhas)
divisão 3: treino até 2020-10-03 (3,235 linhas), valida de 2020-10-03 a 2022-03-05 (1,078 linhas)
divisão 4: treino até 2022-03-05 (4,313 linhas), valida de 2022-03-05 a 2023-09-25 (1,078 linhas)
divisão 5: treino até 2023-09-25 (5,391 linhas), valida de 2023-09-25 a 2025-02-15 (1,078 linhas)


In [5]:
# Passo 4 - funções: preparação da linha de base, avaliação e comparação, em preprocessing.py:
#   - model_columns(frame): as colunas que o modelo vê (numéricas e categóricas), sem o alvo,
#     o cliente e a data
#   - as_object(block): pd.NA passa a np.nan, que é o nulo que o scikit-learn lê (W3)
#   - prepare_base(train, test): mediana nas numéricas e moda nas categóricas, ajustadas só
#     no treino (linha de base)
#   - fit_predict(prepare, train, test, scaler, encoder, model, select): prepara, codifica
#     (one-hot com handle_unknown="ignore"), escala e treina só no train, e prevê o test
#   - evaluate(prepare, frame, splits, ...): F1 e accuracy em cada divisão, como o
#     held_out_f1 da W3
#   - sem(values): o erro-padrão da média (W3)
#   - compare(runs, reference, metric): cada alternativa contra a referência, divisão a
#     divisão; uma diferença menor do que 2 x SEM é ruído
from preprocessing import (as_object, compare, evaluate, fit_predict, model_columns,
                           prepare_base, sem)

In [6]:
# Passo 5 - linha de base: mediana / moda, one-hot, StandardScaler, sem features novas
baseline = evaluate(prepare_base, frame, splits)

print(f"linha de base: F1 {baseline['f1'].mean():.4f} +- {sem(baseline['f1']):.4f}"
      f" | accuracy {baseline['accuracy'].mean():.4f} +- {sem(baseline['accuracy']):.4f}")
baseline.round(4)

linha de base: F1 0.7321 +- 0.0090 | accuracy 0.7056 +- 0.0035


,f1,accuracy
0,0.7543,0.6929
1,0.7258,0.7106
2,0.7350,0.7124
3,0.7015,0.7032
4,0.7439,0.7087


In [7]:
# Passo 6 - o piso: prever sempre 1 (a classe maioritária) nas mesmas linhas de validação
always_one = pd.DataFrame({
    "f1": [f1_score(frame[TARGET].iloc[v], np.ones(len(v), dtype=int)) for _, v in splits],
    "accuracy": [frame[TARGET].iloc[v].mean() for _, v in splits],
})

pd.DataFrame({"linha de base": baseline.mean(), "prever sempre 1": always_one.mean()}).round(4)

,linha de base,prever sempre 1
f1,0.7321,0.7206
accuracy,0.7056,0.5638


In [8]:
# ===== [2.0] BASE COMUM - fim =====
# Como comparar uma alternativa daqui em diante (pacotes 2.1 a 2.6):
#
#   def prepare_a(train, test):      # ajusta tudo só em `train`; devolve (train, test) sem nulos
#       ...
#   compare({"alternativa A": evaluate(prepare_a, frame, splits)}, baseline)
#
# Para comparar escalas (2.4): evaluate(prepare_base, frame, splits, scaler=None) ou outro scaler.

In [9]:
# ===== [2.1] VALORES IMPOSSÍVEIS - início =====
# Passo 7 - regras de domínio da EDA (passos 9 a 11): True onde o valor é impossível
# As regras estão em preprocessing.py (invalid_errors):
#   - days_since_previous_purchase e prior_mean_gap_days: dias negativos (-4)
#   - prior_mean_purchase_value e spend_365d: negativos ou 1e9
#   - e a regra de duas colunas da EDA: uma média de compras não pode ser maior do que a
#     maior dessas compras (prior_max_purchase_value)
# invalid_flags(frame) dá uma coluna por regra, True onde o valor é impossível (como o
# domain_flags da W3); blank_invalid(frame) passa esses valores a NaN (W3: "blank the cell").
# Não aprende nada com os dados, por isso aplica-se igual ao treino, à validação e ao teste.
from preprocessing import blank_invalid, invalid_errors, invalid_flags

flags = invalid_flags(frame)
print("valores impossíveis por coluna:", flags.sum().to_dict())
print("linhas com algum valor impossível:", int(flags.any(axis=1).sum()), "de", len(frame))

valores impossíveis por coluna: {'days_since_previous_purchase': 16, 'prior_mean_gap_days': 16, 'prior_mean_purchase_value': 77, 'spend_365d': 30}
linhas com algum valor impossível: 139 de 6469


In [10]:
# Passo 8 - manter, passar a nulo ou apagar a linha? (W3, células 63 a 69)
# Nos três casos os nulos são depois preenchidos como na linha de base (mediana / moda).
def prepare_blank(train, test):
    """Valores impossíveis passam a nulo, nas duas partes."""
    return prepare_base(blank_invalid(train), blank_invalid(test))


def prepare_drop(train, test):
    """Apaga do TREINO as linhas com valores impossíveis (W3: drop_invalid_rows).

    A validação nunca perde linhas, porque todas as compras do teste precisam de previsão.
    """
    keep = ~invalid_flags(train).any(axis=1)
    return prepare_base(train[keep], blank_invalid(test))


invalid_runs = {"passar a nulo": evaluate(prepare_blank, frame, splits),
                "apagar a linha (treino)": evaluate(prepare_drop, frame, splits)}
compare(invalid_runs, baseline).round(4)

,f1,accuracy,vs base,SEM,melhor em
passar a nulo,0.734164,0.707607,0.00206,0.0011,4 de 5
apagar a linha (treino),0.734751,0.707421,0.002646,0.001175,5 de 5


In [11]:
# ===== [2.1] VALORES IMPOSSÍVEIS - fim =====
# Passo 9 - o que fica decidido no 2.1:
#   - valores impossíveis -> passar a nulo (blank_invalid); os nulos são tratados no 2.2
#   - as 65 linhas vazias (só cliente e alvo) ficam fora do treino; já estão fora das
#     divisões desde o passo 1, e o test.csv não tem linhas assim
print("linhas do teste sem data:", int(test_csv[DATE].isna().sum()))
print("valores impossíveis no teste (a regra aplica-se igual):",
      int(invalid_flags(test_csv).sum().sum()))

linhas do teste sem data: 0
valores impossíveis no teste (a regra aplica-se igual): 18


In [12]:
# ===== [2.2] VALORES EM FALTA - início =====
# Passo 10 - nulos que sobram depois do 2.1
after_21 = blank_invalid(frame)
nulls = after_21.isna().sum()
nulls[nulls > 0].sort_values(ascending=False).to_frame("nulos")

,nulos
prior_std_gap_days,1413
prior_mean_gap_days,1259
prior_std_purchase_value,1048
prior_mean_purchase_value,874
days_since_previous_purchase,812
prior_min_purchase_value,603
prior_max_purchase_value,603
spend_365d,225
transport_method,196
payment_type,195


In [13]:
# Passo 11 - os nulos estruturais (EDA, passo 12): o valor ainda não pode existir porque
# o cliente tem poucas compras anteriores. Mínimo de compras anteriores para cada coluna existir:
needs_history = {
    "days_since_previous_purchase": 1,
    "prior_mean_purchase_value": 1,
    "prior_min_purchase_value": 1,
    "prior_max_purchase_value": 1,
    "prior_std_purchase_value": 2,   # não há desvio-padrão de um só valor
    "prior_mean_gap_days": 2,        # um intervalo precisa de duas compras
    "prior_std_gap_days": 3,         # o desvio dos intervalos precisa de dois intervalos
}


def zero_where_none(frame):
    """Zero onde o valor ainda não existe; os outros nulos ficam para a mediana.

    É o plano "zero where none / median elsewhere" da W3 (célula 79). Não aprende
    nada com os dados, por isso aplica-se igual ao treino e ao teste.
    """
    out = frame.copy()
    for column, minimum in needs_history.items():
        no_history = out[column].isna() & (out["prior_purchase_count"] < minimum)
        out.loc[no_history, column] = 0
    return out


def own_level(frame):
    """Nulos das categóricas passam a um nível próprio, "(missing)" (W3, célula 79)."""
    out = frame.copy()
    _, categorical = model_columns(out)
    out[categorical] = out[categorical].fillna("(missing)")
    return out


structural = int(after_21.isna().sum().sum() - zero_where_none(after_21).isna().sum().sum())
print(f"nulos no total: {int(after_21.isna().sum().sum()):,}, dos quais estruturais: {structural:,}")

nulos no total: 7,617, dos quais estruturais: 5,921


In [14]:
# Passo 12 - comparar planos de preenchimento (W3, células 79 a 87), todos depois do 2.1
def prepare_zero_where_none(train, test):
    return prepare_base(zero_where_none(blank_invalid(train)), zero_where_none(blank_invalid(test)))


def prepare_own_level(train, test):
    return prepare_base(own_level(blank_invalid(train)), own_level(blank_invalid(test)))


fill_runs = {"mediana / moda": invalid_runs["passar a nulo"],
             "zero sem histórico, mediana no resto": evaluate(prepare_zero_where_none, frame, splits),
             "mediana / nível próprio": evaluate(prepare_own_level, frame, splits)}
compare(fill_runs, invalid_runs["passar a nulo"]).round(4)

,f1,accuracy,vs base,SEM,melhor em
mediana / moda,0.734164,0.707607,0.0,0.0,0 de 5
"zero sem histórico, mediana no resto",0.734237,0.707421,0.000074,0.000404,2 de 5
mediana / nível próprio,0.734468,0.707607,0.000304,0.001256,2 de 5


In [15]:
# ===== [2.2] VALORES EM FALTA - fim =====
# Passo 13 - o que fica decidido no 2.1 e no 2.2, numa só preparação para os pacotes seguintes:
#   1. valores impossíveis -> nulo (blank_invalid), igual no treino e no teste
#   2. nulos -> mediana nas numéricas e moda nas categóricas, ajustadas só no treino
#      (as alternativas ficaram dentro do ruído; como na W3, fica a mais simples)
prepare_clean = prepare_blank

clean = invalid_runs["passar a nulo"]
print(f"2.1 + 2.2: F1 {clean['f1'].mean():.4f} +- {sem(clean['f1']):.4f}"
      f" | accuracy {clean['accuracy'].mean():.4f} +- {sem(clean['accuracy']):.4f}")

2.1 + 2.2: F1 0.7342 +- 0.0093 | accuracy 0.7076 +- 0.0024


In [16]:
# ===== [2.4] DISTRIBUIÇÕES E ESCALA - início =====
# Passo 14 - que colunas transformar? Todas as numéricas com cauda longa (EDA, passo 7):
# todas menos random_noise e customer_tenure_days, as duas únicas sem cauda longa.
skewed = [c for c in model_columns(frame)[0] if c not in ["random_noise", "customer_tenure_days"]]
from preprocessing import SKEWED
assert skewed == SKEWED   # a mesma lista que preprocessing.py usa no log1p
print(f"colunas a transformar: {len(skewed)} de {len(model_columns(frame)[0])} numéricas")

# Assimetria (skew) antes e depois do log1p, depois de tirar os valores impossíveis (2.1).
# Só para ver o efeito: aqui nada é ajustado, por isso pode usar o treino inteiro (W3, célula 28).
after_21 = blank_invalid(frame)
pd.DataFrame({"skew antes": after_21[skewed].skew(),
              "skew depois do log1p": np.log1p(after_21[skewed]).skew()}).round(2)

colunas a transformar: 27 de 29 numéricas


,skew antes,skew depois do log1p
Total a Pagar,4.21,-0.48
Total Bruto,4.31,-0.43
Total Liquido,4.15,-0.45
Total Imp_,4.84,-0.76
Total Portes,19.55,2.27
Total Desc_ Global,27.46,9.28
Total Desc_ Linha,5.92,0.24
Custo,4.61,-1.02
N_ Detalhes,2.98,0.08
invoice_count_current_purchase,6.44,5.00


In [17]:
# Passo 15 - os dois tratamentos, sempre pela ordem: valores impossíveis -> tratamento -> nulos.
# Depois do blank_invalid já não há negativos nestas colunas, por isso o log1p não precisa
# do clip(lower=0) que a W3 usa no log1p_of.
# prepare_log1p (preprocessing.py): blank_invalid -> np.log1p nas colunas SKEWED ->
# prepare_base (W3, célula 100: log1p_of). Não aprende nada com os dados, por isso
# aplica-se igual ao treino e ao teste.
from preprocessing import prepare_log1p


def prepare_winsor(train, test):
    """Corta cada coluna aos percentis 1% e 99% do TREINO (W3, célula 100: winsorize).

    Os limites aprendem-se só no `train` e aplicam-se iguais ao `test`.
    """
    train, test = blank_invalid(train), blank_invalid(test)
    low, high = train[skewed].quantile(0.01), train[skewed].quantile(0.99)
    train[skewed] = train[skewed].clip(low, high, axis=1)
    test[skewed] = test[skewed].clip(low, high, axis=1)
    return prepare_base(train, test)

In [18]:
# Passo 16 - comparar os tratamentos com a referência `clean` (2.1 + 2.2), nas mesmas divisões
# e com o mesmo StandardScaler (W3, células 100 a 104). Só conta se "vs base" > 2 x SEM.
treatment_runs = {"log1p": evaluate(prepare_log1p, frame, splits),
                  "winsorize 1% / 99%": evaluate(prepare_winsor, frame, splits)}
treatment_table = compare(treatment_runs, clean)
treatment_table["2 x SEM"] = 2 * treatment_table["SEM"]
treatment_table.round(4)

,f1,accuracy,vs base,SEM,melhor em,2 x SEM
log1p,0.739642,0.707236,0.005478,0.002904,4 de 5,0.005808
winsorize 1% / 99%,0.737703,0.709647,0.003539,0.002745,3 de 5,0.005491


In [19]:
# Passo 17 - comparar escalas com a melhor média do passo 3, o log1p (W3, células 112 a 114).
# A referência continua a ser `clean`, para todas as linhas se lerem contra o mesmo número.
# Sem scaler, a regressão logística pára no limite de 1000 iterações sem convergir: daí os
# avisos ConvergenceWarning em baixo (a W3 mostra o mesmo).
scaler_runs = {"log1p + StandardScaler": treatment_runs["log1p"],
               "log1p + MinMaxScaler": evaluate(prepare_log1p, frame, splits, scaler=MinMaxScaler()),
               "log1p + RobustScaler": evaluate(prepare_log1p, frame, splits, scaler=RobustScaler()),
               "log1p sem scaler": evaluate(prepare_log1p, frame, splits, scaler=None)}
scaler_table = compare(scaler_runs, clean)
scaler_table["2 x SEM"] = 2 * scaler_table["SEM"]
scaler_table.round(4)

,f1,accuracy,vs base,SEM,melhor em,2 x SEM
log1p + StandardScaler,0.739642,0.707236,0.005478,0.002904,4 de 5,0.005808
log1p + MinMaxScaler,0.748076,0.71577,0.013912,0.002819,5 de 5,0.005637
log1p + RobustScaler,0.743428,0.713173,0.009264,0.002626,5 de 5,0.005251
log1p sem scaler,0.744505,0.711688,0.010341,0.004264,4 de 5,0.008527


In [20]:
# Passo 18 - separar as duas mudanças: a tabela anterior compara tudo com `clean`, que não tem
# log1p e usa StandardScaler. Aqui cada mudança é medida sozinha, nas mesmas divisões.
clean_minmax = evaluate(prepare_clean, frame, splits, scaler=MinMaxScaler())
for metric in ["f1", "accuracy"]:
    pair_table = pd.concat([
        compare({"MinMax vs Standard (ambos com log1p)": scaler_runs["log1p + MinMaxScaler"]},
                treatment_runs["log1p"], metric=metric),
        compare({"log1p vs sem log1p (ambos com MinMax)": scaler_runs["log1p + MinMaxScaler"]},
                clean_minmax, metric=metric),
    ])
    pair_table["2 x SEM"] = 2 * pair_table["SEM"]
    print(f"[{metric}]")
    print(pair_table[[metric, "vs base", "SEM", "2 x SEM", "melhor em"]].round(4).to_string(), "\n")

[f1]
                                             f1   vs base       SEM   2 x SEM melhor em
MinMax vs Standard (ambos com log1p)   0.748076  0.008435  0.003569  0.007138    4 de 5
log1p vs sem log1p (ambos com MinMax)  0.748076  0.007298  0.001651  0.003302    5 de 5 

[accuracy]
                                      accuracy   vs base      SEM  2 x SEM melhor em
MinMax vs Standard (ambos com log1p)   0.71577  0.008534  0.00335   0.0067    4 de 5
log1p vs sem log1p (ambos com MinMax)  0.71577   0.00538  0.00291  0.00582    3 de 5 



In [21]:
# ===== [2.4] DISTRIBUIÇÕES E ESCALA - fim =====
# Passo 19 - o que fica decidido no 2.4 (números contra `clean`, F1 0.7342):
#   1. tratamento -> log1p nas 27 colunas com cauda longa
#      - sozinho (com StandardScaler) fica no limite do ruído: +0.0055, com 2 x SEM = 0.0058;
#        o winsorize fica abaixo (+0.0035, 2 x SEM = 0.0055)
#      - o log1p não aprende nada com os dados (não há fuga) e é o tratamento da W3
#   2. escala -> MinMaxScaler
#      - log1p + MinMaxScaler: F1 0.7481, +0.0139 com 2 x SEM = 0.0056, melhor em 5 de 5
#        divisões: é a única diferença claramente acima do ruído
#      - RobustScaler também passa os 2 x SEM (+0.0093 contra 0.0053), mas fica abaixo
#      - sem scaler o modelo não converge, por isso fica de fora mesmo com boa média
#   3. as duas mudanças medidas em separado (passo 18):
#      - MinMax contra Standard, ambos com log1p: F1 +0.0084 (2 x SEM = 0.0071) e
#        accuracy +0.0085 (2 x SEM = 0.0067): melhor nas duas métricas
#      - log1p contra sem log1p, ambos com MinMax: F1 +0.0073 (2 x SEM = 0.0033);
#        na accuracy +0.0054 fica no limite do ruído (2 x SEM = 0.0058)
#      - o log1p sozinho com StandardScaler não melhora a accuracy (-0.0004): só ajuda
#        com o MinMaxScaler
#   Nota: o scaler não faz parte do `prepare`; passa-se a evaluate / fit_predict com
#   scaler=SCALER_24. A célula de submissão ainda usa a preparação do 2.2 (juntar no 2.7).
prepare_24 = prepare_log1p
SCALER_24 = MinMaxScaler()

result_24 = scaler_runs["log1p + MinMaxScaler"]
print(f"2.4: F1 {result_24['f1'].mean():.4f} +- {sem(result_24['f1']):.4f}"
      f" | accuracy {result_24['accuracy'].mean():.4f} +- {sem(result_24['accuracy']):.4f}")

2.4: F1 0.7481 +- 0.0086 | accuracy 0.7158 +- 0.0052


In [22]:
# ===== [2.6] NOVAS FEATURES - início =====
# Passo 20 - as features candidatas. Todas usam só informação que existe no momento da compra.
# Nas divisões, um denominador zero passa a nulo (W3, célula 28: .replace(0, np.nan));
# esse nulo é depois preenchido pela mediana do treino no prepare_base.
new_features = {
    # a primeira compra comporta-se de outra forma (EDA, passo 12: 33% contra 58% de recompra)
    "is_first_purchase": lambda t: (t["prior_purchase_count"] == 0).astype(int),
    # atraso face ao intervalo habitual do cliente
    "days_vs_habit": lambda t: t["days_since_previous_purchase"]
                               / t["prior_mean_gap_days"].replace(0, np.nan),
    # valor médio por compra no último ano
    "avg_ticket_365d": lambda t: t["spend_365d"] / t["purchase_count_365d"].replace(0, np.nan),
    # margem da compra
    "margin": lambda t: t["Total Liquido"] - t["Custo"],
    # peso do desconto na compra
    "discount_share": lambda t: t["Total Desc_ Linha"] / t["Total Bruto"].replace(0, np.nan),
}


def add_features(frame, names=tuple(new_features)):
    """Acrescenta as features `names` a uma tabela. Não aprende nada com os dados."""
    out = frame.copy()
    for name in names:
        out[name] = new_features[name](out)
    return out


after_26 = add_features(blank_invalid(frame))
after_26[list(new_features)].describe().T[["count", "min", "50%", "max"]].round(3)

,count,min,50%,max
is_first_purchase,6469.0,0.000,0.000,1.000
days_vs_habit,5031.0,0.001,0.831,478.667
avg_ticket_365d,5434.0,3.310,720.128,66539.157
margin,6469.0,-1592.119,197.205,11075.600
discount_share,6469.0,0.000,0.046,0.975


In [23]:
# Passo 21 - uma preparação para cada conjunto de features, sempre pela ordem
# blank_invalid -> add_features -> prepare_base: as features calculam-se antes de preencher
# os nulos, para não usarem valores inventados.
def prepare_with(names):
    """Devolve uma função prepare(train, test) = prepare_clean + as features `names`."""
    def prepare(train, test):
        train = add_features(blank_invalid(train), names)
        test = add_features(blank_invalid(test), names)
        return prepare_base(train, test)
    return prepare

In [24]:
# Passo 22 - cada feature sozinha, acrescentada a prepare_clean, contra `clean`
# (W4, "Combination features", passos 9 e 10: paired_delta). Só conta se "vs base" > 2 x SEM.
# A última linha junta as cinco, só para ver se juntas fazem o que sozinhas não fazem.
feature_runs = {name: evaluate(prepare_with([name]), frame, splits) for name in new_features}
feature_runs["todas as 5"] = evaluate(prepare_with(list(new_features)), frame, splits)
feature_table = compare(feature_runs, clean)
feature_table["2 x SEM"] = 2 * feature_table["SEM"]
feature_table.round(4)

,f1,accuracy,vs base,SEM,melhor em,2 x SEM
is_first_purchase,0.734286,0.707607,0.000122,0.00075,1 de 5,0.0015
days_vs_habit,0.734271,0.707421,0.000107,0.00069,2 de 5,0.001379
avg_ticket_365d,0.733388,0.706865,-0.000776,0.000222,0 de 5,0.000444
margin,0.734217,0.707792,0.000053,0.000306,2 de 5,0.000611
discount_share,0.736052,0.709091,0.001888,0.001013,4 de 5,0.002027
todas as 5,0.736541,0.708349,0.002378,0.001351,4 de 5,0.002701


In [25]:
# Passo 23 - confirmação (não pedida no enunciado do 2.6): as mesmas features por cima da
# receita do 2.4 (log1p + MinMaxScaler), contra `result_24`. Ajuda a decidir no 2.7.
def prepare_with_24(names):
    """prepare_24 + as features `names`; o log1p só mexe nas colunas originais (`skewed`)."""
    def prepare(train, test):
        train = add_features(blank_invalid(train), names)
        test = add_features(blank_invalid(test), names)
        train[skewed], test[skewed] = np.log1p(train[skewed]), np.log1p(test[skewed])
        return prepare_base(train, test)
    return prepare


feature_runs_24 = {name: evaluate(prepare_with_24([name]), frame, splits, scaler=SCALER_24)
                   for name in new_features}
feature_table_24 = compare(feature_runs_24, result_24)
feature_table_24["2 x SEM"] = 2 * feature_table_24["SEM"]
feature_table_24.round(4)

,f1,accuracy,vs base,SEM,melhor em,2 x SEM
is_first_purchase,0.746513,0.714286,-0.001563,0.000773,1 de 5,0.001546
days_vs_habit,0.748472,0.716327,0.000396,0.000597,2 de 5,0.001194
avg_ticket_365d,0.748189,0.715955,0.000113,0.000489,2 de 5,0.000978
margin,0.748568,0.716512,0.000492,0.000282,3 de 5,0.000565
discount_share,0.747299,0.715213,-0.000777,0.00083,2 de 5,0.00166


In [26]:
# ===== [2.6] NOVAS FEATURES - fim =====
# Passo 24 - o que fica decidido no 2.6 (números contra `clean`, F1 0.7342):
#   - nenhuma feature passa os 2 x SEM, por isso nenhuma fica:
#       is_first_purchase +0.0001, 2 x SEM = 0.0015, melhor em 1 de 5
#       days_vs_habit     +0.0001, 2 x SEM = 0.0014, melhor em 2 de 5
#       avg_ticket_365d   -0.0008, 2 x SEM = 0.0004, melhor em 0 de 5   <- pior em todas as divisões
#       margin            +0.0001, 2 x SEM = 0.0006, melhor em 2 de 5
#       discount_share    +0.0019, 2 x SEM = 0.0020, melhor em 4 de 5   <- a mais perto, mas ainda dentro do ruído
#   - o passo "testar juntas as que ganham" fica vazio; mesmo as cinco juntas ficam no ruído
#     (+0.0024, 2 x SEM = 0.0027, melhor em 4 de 5)
#   #   - por cima do 2.4 (passo 23) nenhuma melhora; is_first_purchase até piora, por muito
#     pouco (-0.0016, 2 x SEM = 0.0015)git add notebooks/02_preprocessing.ipynb
#   - porquê: a informação já está nas colunas que existem (por exemplo, is_first_purchase
#     é só prior_purchase_count == 0, e prior_purchase_count já entra no modelo)
#   Como as alternativas empatam, fica a mais simples: sem features novas.
prepare_26 = prepare_clean

In [27]:
# ===== [2.3] CATEGÓRICAS - início =====
# Passo 25 - o que há para codificar (EDA, passos 14 e 15). Os níveis raros não se juntam
# num nível "outros": as aulas só o referem, não o fazem. O que fazem é o que se testa a
# seguir: handle_unknown no encoder, codificações compactas (count e target, W4 passo 5)
# e, para uma coluna com demasiados níveis, deixá-la de fora (W4 de regressão, Step 3).
_, categorical = model_columns(frame)
pd.DataFrame({
    "níveis": frame[categorical].nunique(),
    "níveis com menos de 30 linhas": [int((frame[c].value_counts() < 30).sum())
                                      for c in categorical],
    "nulos": frame[categorical].isna().sum(),
}).sort_values("níveis", ascending=False)

,níveis,níveis com menos de 30 linhas,nulos
transport_method,80,51,196
payment_type,9,4,195
salesperson,9,3,194
commercial_zone,5,2,0
document_series,3,0,0
warehouse,2,1,195


In [28]:
# Passo 26 - em cada divisão: quantas colunas o one-hot cria (W4, passo 8) e quantas linhas
# de validação têm um nível que o treino não viu. É por estas que o handle_unknown existe.
width_rows = []
for fold, (train_idx, val_idx) in enumerate(splits, start=1):
    train, test = prepare_clean(frame.iloc[train_idx], frame.iloc[val_idx])
    unseen = pd.concat([~test[c].isin(train[c]) for c in categorical], axis=1)
    width_rows.append({
        "divisão": fold,
        "colunas one-hot": OneHotEncoder(handle_unknown="ignore")
                           .fit(as_object(train[categorical])).get_feature_names_out().size,
        "das quais transport_method": train["transport_method"].nunique(),
        "linhas de validação com nível novo": int(unseen.any(axis=1).sum()),
        "em transport_method": int(unseen["transport_method"].sum()),
    })
pd.DataFrame(width_rows).set_index("divisão")

,colunas one-hot,das quais transport_method,linhas de validação com nível novo,em transport_method
divisão,,,,
1,56,42,178,13
2,67,51,132,132
3,80,62,634,219
4,94,71,181,22
5,104,76,27,27


In [29]:
# Passo 27 - comparar codificações por cima de prepare_clean (2.1 + 2.2), contra `clean`,
# nas mesmas divisões e com o mesmo StandardScaler (W4, passos 5 a 7). Os mesmos quatro
# encoders da W4, com os mesmos parâmetros. Só conta se "vs base" > 2 x SEM.
# - ordinal, count e target transformam um nível num número, que depois passa pelo scaler,
#   como na W4; o target encoding usa o alvo, por isso é ajustado só no treino da divisão
# - com drop="first", um nível que só aparece na validação fica a zeros, igual ao nível
#   retirado; o scikit-learn avisa disso (UserWarning) em cada divisão, por isso o aviso é
#   silenciado, como a W4 faz com os seus avisos
warnings.filterwarnings("ignore", category=UserWarning)

encoders = {
    "one-hot, drop='first'": OneHotEncoder(handle_unknown="ignore", drop="first",
                                           sparse_output=False),
    "ordinal": OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
    "count": CountEncoder(normalize=True, handle_unknown=0, handle_missing="value"),
    "target": TargetEncoder(handle_unknown="value", handle_missing="value"),
}


def prepare_without_transport(train, test):
    """prepare_clean sem transport_method, a coluna com 80 níveis."""
    return prepare_clean(train.drop(columns="transport_method"),
                         test.drop(columns="transport_method"))


encoding_runs = {name: evaluate(prepare_clean, frame, splits, encoder=encoder)
                 for name, encoder in encoders.items()}
encoding_runs["one-hot, sem transport_method"] = evaluate(prepare_without_transport,
                                                          frame, splits)
for metric in ["f1", "accuracy"]:
    encoding_table = compare(encoding_runs, clean, metric=metric)
    encoding_table["2 x SEM"] = 2 * encoding_table["SEM"]
    print(f"[{metric}]")
    print(encoding_table.round(4).to_string(), "\n")

[f1]
                                     f1  accuracy   vs base       SEM melhor em   2 x SEM
one-hot, drop='first'          0.734485  0.708163  0.000321  0.001204    2 de 5  0.002407
ordinal                        0.741409  0.718738  0.007245  0.002307    5 de 5  0.004614
count                          0.730803  0.712987 -0.003361  0.006088    3 de 5  0.012177
target                         0.733991  0.712801 -0.000173  0.003446    2 de 5  0.006893
one-hot, sem transport_method  0.735859  0.717069  0.001695   0.00437    3 de 5  0.008741 

[accuracy]
                               accuracy   vs base       SEM melhor em   2 x SEM
one-hot, drop='first'          0.708163  0.000557  0.001775    3 de 5  0.003549
ordinal                        0.718738  0.011132   0.00231    5 de 5   0.00462
count                          0.712987   0.00538  0.003096    4 de 5  0.006192
target                         0.712801  0.005195  0.002761    4 de 5  0.005522
one-hot, sem transport_method  0.717069  0

In [30]:
# Passo 28 - confirmação, como no 2.6: as mesmas codificações por cima da receita do 2.4
# (log1p + MinMaxScaler), contra `result_24`. Ajuda a decidir no 2.7.
def prepare_24_without_transport(train, test):
    """prepare_24 sem transport_method."""
    return prepare_24(train.drop(columns="transport_method"),
                      test.drop(columns="transport_method"))


encoding_runs_24 = {name: evaluate(prepare_24, frame, splits, scaler=SCALER_24, encoder=encoder)
                    for name, encoder in encoders.items()}
encoding_runs_24["one-hot, sem transport_method"] = evaluate(
    prepare_24_without_transport, frame, splits, scaler=SCALER_24)
for metric in ["f1", "accuracy"]:
    encoding_table_24 = compare(encoding_runs_24, result_24, metric=metric)
    encoding_table_24["2 x SEM"] = 2 * encoding_table_24["SEM"]
    print(f"[{metric}]")
    print(encoding_table_24.round(4).to_string(), "\n")

[f1]
                                     f1  accuracy   vs base       SEM melhor em   2 x SEM
one-hot, drop='first'          0.746571  0.716141 -0.001505   0.00293    2 de 5   0.00586
ordinal                        0.748861  0.717254  0.000785  0.005354    3 de 5  0.010708
count                          0.743285  0.718553 -0.004791  0.005319    3 de 5  0.010639
target                         0.749137  0.719852  0.001061  0.003108    2 de 5  0.006216
one-hot, sem transport_method  0.752546  0.724304   0.00447  0.003436    4 de 5  0.006872 

[accuracy]
                               accuracy   vs base       SEM melhor em   2 x SEM
one-hot, drop='first'          0.716141  0.000371  0.002465    2 de 5   0.00493
ordinal                        0.717254  0.001484  0.004644    4 de 5  0.009288
count                          0.718553  0.002783  0.001901    4 de 5  0.003802
target                         0.719852  0.004082  0.004781    1 de 5  0.009561
one-hot, sem transport_method  0.724304  0

In [31]:
# ===== [2.3] CATEGÓRICAS - fim =====
# Passo 29 - o que fica decidido no 2.3:
#   1. níveis raros -> não se juntam (as aulas não o fazem); os níveis que só aparecem na
#      validação ficam a zeros (handle_unknown="ignore", passo 26)
#   2. codificação -> one-hot, como na linha de base
#      - contra `clean` (passo 27), o ordinal é o único acima do ruído: F1 +0.0072
#        (2 x SEM = 0.0046) e accuracy +0.0111 (2 x SEM = 0.0046), melhor em 5 de 5
#      - mas por cima da receita do 2.4 (passo 28), que é a preparação a integrar no 2.7,
#        essa vantagem desaparece: F1 +0.0008 com 2 x SEM = 0.0107; nenhuma codificação
#        passa os 2 x SEM no F1
#      - o ordinal impõe uma ordem que os níveis não têm (W4, passo 5); como empatam na
#        receita do 2.4, fica a mais simples: one-hot
#      - drop="first", count e target ficam dentro do ruído nas duas comparações
#   3. transport_method fica
#      - sem ela, o F1 fica dentro do ruído nas duas comparações (+0.0017 e +0.0045)
#      - a accuracy melhora (+0.0095, 2 x SEM = 0.0061; por cima do 2.4 +0.0085, acima
#        do ruído por pouco, 2 x SEM = 0.0082): candidata a sair na seleção de features
#        (tarefa 3), que depende da métrica escolhida
prepare_23 = prepare_clean
ENCODER_23 = None   # None = o one-hot do fit_predict
print("2.3: sem alterações à preparação; a codificação continua a ser o one-hot")

2.3: sem alterações à preparação; a codificação continua a ser o one-hot


In [32]:
# ===== [2.5] CLIENTES.CSV - início =====
# Passo 30 - ler clientes.csv como na EDA (passo 21), com customer_id no índice, e escolher
# as candidatas. Os níveis e nulos contam só os 610 clientes do treino, como na EDA.
clientes = pd.read_csv("../data/clientes.csv", index_col="customer_id")
client_columns = ["customer_since_year", "country_or_region", "commercial_region"]

print(f"{len(clientes):,} clientes, {clientes.index.nunique():,} customer_id distintos")
print("clientes do treino em clientes.csv:", raw[GROUP].isin(clientes.index).all(),
      "| clientes do test.csv em clientes.csv:", test_csv[GROUP].isin(clientes.index).all())

in_train = clientes[clientes.index.isin(raw[GROUP])]
reasons = {
    "customer_type": "fica de fora: vazia",
    "company_name": "fica de fora: identifica o cliente (W3, célula 61)",
    "address": "fica de fora: identifica o cliente (W3, célula 61)",
    "island": "fica de fora: falta em quase metade dos clientes",
}
for column in ["postal_code", "street", "city", "postal_area", "municipality"]:
    reasons[column] = "fica de fora: demasiados níveis (W4 de regressão, Step 3)"
for column in client_columns:
    reasons[column] = "candidata"
pd.DataFrame({"% nulos": (100 * in_train.isna().mean()).round(1),
              "níveis": in_train.nunique(),
              "decisão": pd.Series(reasons)})

1,143 clientes, 1,143 customer_id distintos
clientes do treino em clientes.csv: True | clientes do test.csv em clientes.csv: True


,% nulos,níveis,decisão
address,0.0,600,"fica de fora: identifica o cliente (W3, célula..."
city,38.7,260,fica de fora: demasiados níveis (W4 de regress...
commercial_region,2.0,32,candidata
company_name,0.0,609,"fica de fora: identifica o cliente (W3, célula..."
country_or_region,0.0,7,candidata
customer_since_year,0.0,10,candidata
customer_type,100.0,0,fica de fora: vazia
island,46.9,10,fica de fora: falta em quase metade dos clientes
municipality,6.7,150,fica de fora: demasiados níveis (W4 de regress...
postal_area,6.6,189,fica de fora: demasiados níveis (W4 de regress...


In [33]:
# Passo 31 - juntar as colunas de cada cliente às suas compras. O merge não aparece nas
# práticas: aqui é uma consulta por índice com .loc, porque customer_id é o índice de
# clientes. A junção não aprende nada com os dados, por isso aplica-se igual ao treino,
# à validação e ao test.csv, sem fuga.
def add_client_columns(frame, columns):
    """Acrescenta a cada compra as colunas `columns` do seu cliente, lidas de clientes.csv."""
    out = frame.copy()
    for column in columns:   # coluna a coluna, para cada uma manter o seu tipo
        out[column] = clientes.loc[out[GROUP], column].to_numpy()
    return out


joined = add_client_columns(frame, client_columns)
print("linhas antes e depois da junção:", len(frame), len(joined))
print("nulos depois da junção:", joined[client_columns].isna().sum().to_dict())

# customer_since_year não pode ser posterior à compra (seria informação do futuro). As datas
# comparam-se como texto, como na EDA: "2018" vem antes de "2018-05-01" e depois de "2017-12-31".
print("compras com customer_since_year depois da compra:",
      int((joined["customer_since_year"].astype(str) > joined[DATE]).sum()))
# repete customer_tenure_days? (redundante acima de 0,70, como no passo 18 da EDA)
print(f"|r| entre customer_since_year e customer_tenure_days:"
      f" {abs(joined['customer_since_year'].corr(joined['customer_tenure_days'])):.2f}")

linhas antes e depois da junção: 6469 6469
nulos depois da junção: {'customer_since_year': 0, 'country_or_region': 0, 'commercial_region': 35}
compras com customer_since_year depois da compra: 0
|r| entre customer_since_year e customer_tenure_days: 0.32


In [34]:
# Passo 32 - cada coluna sozinha e as três juntas, por cima de prepare_clean (2.1 + 2.2),
# contra `clean`, como no 2.6 (W4, "Combination features", passos 9 e 10). Os nulos que a
# junção traz (commercial_region) são preenchidos pela moda do treino (2.2) e as categóricas
# vão para o one-hot (2.3). Só conta se "vs base" > 2 x SEM.
def prepare_with_clients(columns, prepare=prepare_clean):
    """Devolve uma função prepare(train, test) = `prepare` + as colunas `columns` de clientes."""
    def prepare_joined(train, test):
        return prepare(add_client_columns(train, columns), add_client_columns(test, columns))
    return prepare_joined


client_runs = {column: evaluate(prepare_with_clients([column]), frame, splits)
               for column in client_columns}
client_runs["as 3 juntas"] = evaluate(prepare_with_clients(client_columns), frame, splits)
for metric in ["f1", "accuracy"]:
    client_table = compare(client_runs, clean, metric=metric)
    client_table["2 x SEM"] = 2 * client_table["SEM"]
    print(f"[{metric}]")
    print(client_table.round(4).to_string(), "\n")

[f1]
                           f1  accuracy   vs base       SEM melhor em   2 x SEM
customer_since_year  0.727414  0.708163  -0.00675  0.001731    0 de 5  0.003461
country_or_region    0.733152  0.706308 -0.001012  0.000627    0 de 5  0.001255
commercial_region    0.732812  0.705566 -0.001352  0.001278    1 de 5  0.002557
as 3 juntas          0.725434  0.705751  -0.00873  0.001999    0 de 5  0.003998 

[accuracy]
                     accuracy   vs base       SEM melhor em   2 x SEM
customer_since_year  0.708163  0.000557  0.003221    1 de 5  0.006443
country_or_region    0.706308 -0.001299   0.00086    0 de 5  0.001721
commercial_region    0.705566 -0.002041  0.001188    1 de 5  0.002376
as 3 juntas          0.705751 -0.001855  0.002705    2 de 5  0.005409 



In [35]:
# Passo 33 - confirmação, como no 2.3 e no 2.6: as mesmas colunas por cima da receita do 2.4
# (log1p + MinMaxScaler), contra `result_24`. O log1p só mexe nas colunas originais
# (`skewed`), por isso customer_since_year entra sem transformação.
client_runs_24 = {column: evaluate(prepare_with_clients([column], prepare_24), frame, splits,
                                   scaler=SCALER_24)
                  for column in client_columns}
client_runs_24["as 3 juntas"] = evaluate(prepare_with_clients(client_columns, prepare_24),
                                         frame, splits, scaler=SCALER_24)
for metric in ["f1", "accuracy"]:
    client_table_24 = compare(client_runs_24, result_24, metric=metric)
    client_table_24["2 x SEM"] = 2 * client_table_24["SEM"]
    print(f"[{metric}]")
    print(client_table_24.round(4).to_string(), "\n")

[f1]
                           f1  accuracy   vs base       SEM melhor em   2 x SEM
customer_since_year  0.744235  0.717996 -0.003841   0.00245    2 de 5  0.004899
country_or_region    0.747492  0.715584 -0.000584   0.00068    2 de 5  0.001361
commercial_region    0.745636  0.712801 -0.002441  0.002283    2 de 5  0.004566
as 3 juntas          0.741504  0.715955 -0.006572  0.003175    1 de 5  0.006349 

[accuracy]
                     accuracy   vs base       SEM melhor em   2 x SEM
customer_since_year  0.717996  0.002226   0.00175    2 de 5  0.003501
country_or_region    0.715584 -0.000186  0.000899    3 de 5  0.001799
commercial_region    0.712801 -0.002968  0.003192    2 de 5  0.006384
as 3 juntas          0.715955  0.000186  0.002203    2 de 5  0.004406 



In [36]:
# ===== [2.5] CLIENTES.CSV - fim =====
# Passo 34 - o que fica decidido no 2.5: clientes.csv não entra.
#   1. a junção funciona sem falhas: todos os clientes do treino e do test.csv estão na
#      tabela, uma linha por cliente; customer_since_year nunca é posterior à compra e não
#      repete customer_tenure_days (|r| = 0.32)
#   2. das 12 colunas, só 3 eram utilizáveis (passo 30); as outras estão vazias, identificam
#      o cliente, têm demasiados níveis ou faltam em quase metade dos clientes
#   3. nenhuma das 3 melhora o modelo (passo 32, contra `clean`, F1):
#        customer_since_year  -0.0068, 2 x SEM = 0.0035, melhor em 0 de 5   <- pior
#        country_or_region    -0.0010, 2 x SEM = 0.0013                     <- ruído
#        commercial_region    -0.0014, 2 x SEM = 0.0026                     <- ruído
#        as 3 juntas          -0.0087, 2 x SEM = 0.0040, melhor em 0 de 5   <- pior
#      na accuracy ficam todas dentro do ruído
#   4. por cima da receita do 2.4 (passo 33) também nenhuma ajuda; as 3 juntas pioram o F1
#      (-0.0066, 2 x SEM = 0.0063)
#   - porquê: o que a tabela diz de cada cliente já está nas compras (há quanto tempo é
#     cliente, quanto e quando compra), e a região não se liga à recompra
prepare_25 = prepare_clean
print("2.5: clientes.csv não entra; a preparação fica igual")

2.5: clientes.csv não entra; a preparação fica igual


In [37]:
# ===== [2.7] INTEGRAÇÃO FINAL - início =====
# Passo 35 - a preparação final junta as decisões do 2.1 ao 2.6. Só o 2.1, o 2.2 e o 2.4 mudam
# a preparação; o 2.3, o 2.5 e o 2.6 mantiveram o que já existia (one-hot, sem clientes.csv,
# sem features novas). A transport_method fica por agora: decide-se na seleção (tarefa 3).
# Em preprocessing.py: prepare_final = prepare_log1p, ENCODER_FINAL = None, SCALER_FINAL =
# MinMaxScaler(). Os asserts confirmam que são as decisões do 2.3 e do 2.4.
from preprocessing import ENCODER_FINAL, SCALER_FINAL, prepare_final
assert prepare_final is prepare_24      # valores impossíveis -> nulo, log1p, mediana / moda
assert ENCODER_FINAL is ENCODER_23      # None = one-hot com handle_unknown="ignore"
assert type(SCALER_FINAL) is type(SCALER_24)   # MinMaxScaler

final_run = evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL)
for metric in ["f1", "accuracy"]:
    final_table = compare({"2.1 + 2.2 (clean)": clean, "final (2.1 + 2.2 + 2.4)": final_run},
                          baseline, metric=metric)
    final_table["2 x SEM"] = 2 * final_table["SEM"]
    print(f"[{metric}] contra a linha de base do passo 5")
    print(final_table[[metric, "vs base", "SEM", "2 x SEM", "melhor em"]].round(4).to_string(), "\n")

[f1] contra a linha de base do passo 5
                               f1   vs base       SEM   2 x SEM melhor em
2.1 + 2.2 (clean)        0.734164   0.00206    0.0011    0.0022    4 de 5
final (2.1 + 2.2 + 2.4)  0.748076  0.015972  0.001742  0.003484    5 de 5 

[accuracy] contra a linha de base do passo 5
                         accuracy   vs base       SEM   2 x SEM melhor em
2.1 + 2.2 (clean)        0.707607  0.002041  0.001644  0.003288    3 de 5
final (2.1 + 2.2 + 2.4)   0.71577  0.010204  0.001968  0.003936    5 de 5 



In [38]:
# Passo 36 - a preparação final aplicada ao test.csv: ajustada só no treino inteiro (as medianas
# e as modas vêm do treino; o log1p e o blank_invalid não aprendem nada) e aplicada igual ao teste.
train_final, test_final = prepare_final(frame, test_csv)
numeric, categorical = model_columns(train_final)

pd.Series({
    "linhas: treino / teste": f"{len(train_final):,} / {len(test_final):,}",
    "mesmas colunas, pela mesma ordem": list(model_columns(test_final)[0]) == numeric
                                        and list(model_columns(test_final)[1]) == categorical,
    "nulos no treino / teste": f"{train_final[numeric + categorical].isna().sum().sum()}"
                               f" / {test_final[numeric + categorical].isna().sum().sum()}",
    "valores impossíveis no teste: antes / depois do blank_invalid":
        f"{invalid_flags(test_csv).sum().sum()} / {invalid_flags(blank_invalid(test_csv)).sum().sum()}",
    # níveis que o treino não tem: o one-hot põe-nos a zeros (handle_unknown="ignore")
    "linhas do teste com um nível novo, por coluna":
        {c: int((~test_final[c].isin(train_final[c])).sum()) for c in categorical},
}).to_frame("verificação")

,verificação
linhas: treino / teste,"6,469 / 986"
"mesmas colunas, pela mesma ordem",True
nulos no treino / teste,0 / 0
valores impossíveis no teste: antes / depois do blank_invalid,18 / 0
"linhas do teste com um nível novo, por coluna","{'payment_type': 3, 'salesperson': 0, 'commerc..."


In [39]:
# Passo 37 - todas as decisões do pré-processamento numa tabela (base do PDF da tarefa 5).
# Números contra a referência de cada passo, nas mesmas 5 divisões; "ruído" = abaixo de 2 x SEM.
decisions = pd.DataFrame([
    ("2.0", "divisão por tempo, TimeSeriesSplit(5)", "por cliente (GroupKFold)",
     "o test.csv é posterior ao treino e 94.5% das linhas são de clientes conhecidos"),
    ("2.0", "65 linhas sem data ficam de fora", "-",
     "só têm cliente e alvo; o test.csv não tem linhas assim (W5)"),
    ("2.1", "valores impossíveis -> nulo", "manter; apagar a linha",
     "empata com apagar (F1 +0.0006, ruído) e é a única opção aplicável ao test.csv"),
    ("2.2", "nulos -> mediana / moda", "zero sem histórico; nível próprio",
     "todas empatam (ruído); fica a mais simples, como na W3"),
    ("2.3", "one-hot, sem juntar níveis raros", "drop='first', ordinal, count, target",
     "por cima do 2.4 nenhuma passa os 2 x SEM; o ordinal impõe uma ordem (W4)"),
    ("2.3", "transport_method fica (por agora)", "retirá-la",
     "F1 no ruído; accuracy +0.0085 (2 x SEM 0.0082): decidir na tarefa 3"),
    ("2.4", "log1p nas 27 colunas com cauda longa", "manter; winsorize 1% / 99%",
     "com MinMax: F1 +0.0073 (2 x SEM 0.0033); não apaga nenhuma linha"),
    ("2.4", "MinMaxScaler", "Standard, Robust, sem scaler",
     "com log1p: F1 +0.0084, accuracy +0.0085 contra Standard; sem scaler não converge"),
    ("2.5", "clientes.csv não entra", "3 colunas, sozinhas e juntas",
     "nenhuma ajuda; customer_since_year piora o F1 (-0.0068, 0 de 5)"),
    ("2.6", "nenhuma feature nova", "5 features, sozinhas e juntas",
     "nenhuma passa os 2 x SEM: a informação já está nas colunas existentes"),
], columns=["pacote", "decisão", "alternativas testadas", "porquê"])
print(decisions.to_string(index=False))

pacote                               decisão                alternativas testadas                                                                           porquê
   2.0 divisão por tempo, TimeSeriesSplit(5)             por cliente (GroupKFold)   o test.csv é posterior ao treino e 94.5% das linhas são de clientes conhecidos
   2.0      65 linhas sem data ficam de fora                                    -                      só têm cliente e alvo; o test.csv não tem linhas assim (W5)
   2.1           valores impossíveis -> nulo               manter; apagar a linha    empata com apagar (F1 +0.0006, ruído) e é a única opção aplicável ao test.csv
   2.2               nulos -> mediana / moda    zero sem histórico; nível próprio                           todas empatam (ruído); fica a mais simples, como na W3
   2.3      one-hot, sem juntar níveis raros drop='first', ordinal, count, target         por cima do 2.4 nenhuma passa os 2 x SEM; o ordinal impõe uma ordem (W4)
   2.3     transport_m

In [40]:
# ===== [2.7] INTEGRAÇÃO FINAL - fim =====
# Passo 38 - o que as tarefas 3 e 4 usam daqui em diante:
#   - prepare_final, ENCODER_FINAL e SCALER_FINAL: a preparação final, sempre igual
#   - evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL,
#              model=...): a mesma régua, agora com qualquer modelo (model=None = regressão
#              logística)
#   - fit_predict(..., model=...): o mesmo, ajustado no treino inteiro, para prever o test.csv
# Resultado final da tarefa 2: F1 0.7481 e accuracy 0.7158, contra 0.7321 e 0.7056 da linha
# de base (passo 35).
print(f"preparação final: F1 {final_run['f1'].mean():.4f} +- {sem(final_run['f1']):.4f}"
      f" | accuracy {final_run['accuracy'].mean():.4f} +- {sem(final_run['accuracy']):.4f}")

preparação final: F1 0.7481 +- 0.0086 | accuracy 0.7158 +- 0.0052


In [41]:
# ===== [3.0] BASE DA SELEÇÃO - início =====
# Passo 40 - os identificadores, tratados com espírito crítico (pedido do homework):
# nenhuma destas colunas descreve a compra, por isso nenhuma entra na seleção.
identifier_like = pd.DataFrame([
    ("ID", "índice da tabela", "só identifica a linha (e a submissão no Kaggle)"),
    ("customer_id", "OUT_OF_MODEL", "identifica o cliente; como número, o modelo leria"
                                    " o cliente 900000 como 'maior' do que o 100000"),
    ("purchase_date", "OUT_OF_MODEL", "texto; serve para ordenar as divisões, e a informação"
                                      " temporal já está na recência, antiguidade e janelas"),
], columns=["coluna", "onde está", "porque não é feature"])
print(identifier_like.to_string(index=False))

# confirmar que nenhuma chega ao modelo
numeric, categorical = model_columns(frame)
print("\nidentificadores entre as colunas do modelo:",
      [c for c in ["ID", GROUP, DATE] if c in numeric + categorical])
print(f"colunas antes do one-hot: {len(numeric)} numéricas e {len(categorical)} categóricas")
# random_noise parece uma medida, mas não mede nada: é testado no 3.4

       coluna        onde está                                                                                      porque não é feature
           ID índice da tabela                                                           só identifica a linha (e a submissão no Kaggle)
  customer_id     OUT_OF_MODEL           identifica o cliente; como número, o modelo leria o cliente 900000 como 'maior' do que o 100000
purchase_date     OUT_OF_MODEL texto; serve para ordenar as divisões, e a informação temporal já está na recência, antiguidade e janelas

identificadores entre as colunas do modelo: []
colunas antes do one-hot: 29 numéricas e 6 categóricas


In [42]:
# Passo 41 - a referência da tarefa 3: todas as colunas (select=None tem de dar exatamente
# o mesmo que a preparação final) e quantas colunas o modelo vê em cada divisão depois do
# one-hot. É desse número que os seletores do 3.1 a 3.5 tiram colunas.
reference_3 = evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL)
print("igual à preparação final do 2.7:",
      bool((reference_3[["f1", "accuracy"]] == final_run[["f1", "accuracy"]]).all().all()))

widths = []
for train_idx, val_idx in splits:
    train, test = prepare_final(frame.iloc[train_idx], frame.iloc[val_idx])
    numeric, categorical = model_columns(train)
    encoder = OneHotEncoder(handle_unknown="ignore").fit(as_object(train[categorical]))
    widths.append(len(numeric) + encoder.get_feature_names_out().size)
print("colunas que o modelo vê, por divisão:", widths)
print(f"referência da tarefa 3: F1 {reference_3['f1'].mean():.4f} +- {sem(reference_3['f1']):.4f}"
      f" | accuracy {reference_3['accuracy'].mean():.4f}")
# ===== [3.0] BASE DA SELEÇÃO - fim =====
# Como medir um seletor daqui em diante (3.1 a 3.5): o seletor é ajustado só no treino de cada
# divisão, e o resultado traz as colunas que ficaram (`kept`) e quantas são (`colunas`):
#   run = evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL,
#                  select=SelectKBest(..., k=20))
#   compare({"nome": run}, reference_3)

igual à preparação final do 2.7: True


colunas que o modelo vê, por divisão: [85, 96, 109, 123, 133]
referência da tarefa 3: F1 0.7481 +- 0.0086 | accuracy 0.7158


In [43]:
# ===== [3.1] FILTROS - início =====
# Passo 42 - o que os filtros precisam, da W4 ("Feature selection I", células 64, 70 e 84).
# Os imports ficam aqui para não mexer na célula de imports (regra da tarefa 3).
from collections import Counter

from scipy.stats import chi2_contingency, pointbiserialr
from sklearn.feature_selection import SelectKBest, VarianceThreshold

REDUNDANT = 0.70   # acima disto, um par de colunas é o mesmo facto registado duas vezes (W4, EDA)
# O fit_predict põe as numéricas primeiro e o one-hot depois (na W4 é ao contrário)
N_NUMERIC = len(model_columns(frame)[0])


def cramers_v(column, outcome):
    """Qui-quadrado numa tabela de contingência, reescalado para [0, 1] (W4, célula 64)."""
    table = pd.crosstab(column, outcome)
    if min(table.shape) < 2:
        return 0.0, float("nan"), 0.0, float(table.to_numpy().min())
    statistic, p, _, expected = chi2_contingency(table, correction=False)
    v = np.sqrt(statistic / (table.to_numpy().sum() * (min(table.shape) - 1)))
    return statistic, p, v, expected.min()


def encoded_effect_scores(matrix, outcome):
    """Um efeito em [0, 1] por coluna codificada (W4, célula 70).

    |r| ponto-bisserial para as numéricas e V de Cramér para as colunas one-hot. Adaptado à
    ordem do fit_predict: as N_NUMERIC numéricas primeiro, o one-hot depois. O MinMaxScaler
    não muda nenhum dos dois (é uma escala linear) e deixa o one-hot em 0/1.
    """
    matrix = np.asarray(matrix, dtype=float)
    dummy_block = matrix[:, N_NUMERIC:]
    if not np.isin(dummy_block, [0.0, 1.0]).all():
        raise ValueError("o V de Cramér precisa do bloco one-hot em 0/1")
    target = np.asarray(outcome, dtype=int)
    scores = []
    for position in range(matrix.shape[1]):
        values = matrix[:, position]
        if position >= N_NUMERIC:
            effect = cramers_v(pd.Series(values), pd.Series(target))[2]
        elif values.min() == values.max():
            effect = 0.0   # coluna constante neste treino: não se liga a nada
        else:
            effect = abs(pointbiserialr(target, values).statistic)
        scores.append(effect)
    return np.nan_to_num(np.asarray(scores, dtype=float))


class CorrelationFilter:
    """Ordena por relevância e salta as colunas redundantes com uma já escolhida (W4, célula 84).

    Com `redundancy=None` é o SelectKBest; com `k=None` não há limite e fica tudo o que não
    repete uma coluna já escolhida. Acrescentei `transform` e `get_feature_names_out`, que o
    fit_predict e o evaluate do 3.0 usam; a escolha das colunas é a da W4.
    """

    def __init__(self, k, redundancy=None):
        self.k = k
        self.redundancy = redundancy

    def fit(self, frame, outcome):
        self.names_ = np.asarray(frame.columns)
        matrix = frame.to_numpy(dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            relevance = encoded_effect_scores(matrix, outcome)
            between = np.nan_to_num(np.abs(np.corrcoef(matrix, rowvar=False)))
        keep = []
        for candidate in np.argsort(relevance)[::-1]:
            if len(keep) == self.k:
                break
            if self.redundancy is not None and any(
                    between[candidate, chosen] > self.redundancy for chosen in keep):
                continue
            keep.append(int(candidate))
        self.support_ = np.zeros(matrix.shape[1], dtype=bool)
        self.support_[keep] = True
        return self

    def get_support(self):
        return self.support_

    def transform(self, frame):
        return np.asarray(frame)[:, self.support_]

    def get_feature_names_out(self):
        return self.names_[self.support_]


def kept_in_most(run):
    """As colunas mantidas em mais de metade das divisões, com a contagem (W4, `report`)."""
    kept = Counter(column for columns in run["kept"] for column in columns)
    return [f"{column} {n}/{len(run)}" for column, n in kept.most_common() if n > len(run) / 2]


def filter_table(runs, metric=METRIC):
    """compare() contra reference_3, com o número médio de colunas e o 2 x SEM."""
    table = compare(runs, reference_3, metric=metric)
    table.insert(0, "colunas", [runs[name]["colunas"].mean() for name in table.index])
    table["2 x SEM"] = 2 * table["SEM"]
    return table


print(f"{N_NUMERIC} numéricas primeiro; redundância acima de {REDUNDANT}")

29 numéricas primeiro; redundância acima de 0.7


In [44]:
# Passo 43 - VarianceThreshold(0): remove as colunas constantes no treino de cada divisão
# (W4, células 56 a 61). É um detetor de colunas constantes, não uma forma de ordenar.
filter_runs = {"VarianceThreshold(0)": evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL,
                                                encoder=ENCODER_FINAL,
                                                select=VarianceThreshold(0.0))}
removed = [width - n for width, n in zip(widths, filter_runs["VarianceThreshold(0)"]["colunas"])]
print("colunas por divisão:", widths)
print("colunas removidas por divisão:", removed)

colunas por divisão: [85, 96, 109, 123, 133]
colunas removidas por divisão: [0, 0, 0, 0, 0]


In [45]:
# Passo 44 - relevância: SelectKBest com encoded_effect_scores, cada coluna codificada contra o
# alvo (W4, células 69 a 72). k fixo; o modelo vê 85 a 133 colunas consoante a divisão.
K_VALUES = [10, 20, 40, 60, 80]
for k in K_VALUES:
    filter_runs[f"SelectKBest(k={k})"] = evaluate(
        prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL,
        select=SelectKBest(encoded_effect_scores, k=k))
filter_table({name: filter_runs[name] for name in filter_runs if "KBest" in name}).round(4)

,colunas,f1,accuracy,vs base,SEM,melhor em,2 x SEM
SelectKBest(k=10),10.0,0.740249,0.704638,-0.007827,0.001987,0 de 5,0.003975
SelectKBest(k=20),20.0,0.753311,0.718924,0.005235,0.004309,4 de 5,0.008619
SelectKBest(k=40),40.0,0.746214,0.715399,-0.001862,0.004052,2 de 5,0.008104
SelectKBest(k=60),60.0,0.747887,0.718367,-0.000189,0.004401,3 de 5,0.008802
SelectKBest(k=80),80.0,0.744004,0.713915,-0.004072,0.002225,1 de 5,0.00445


In [46]:
# Passo 45 - redundância: o CorrelationFilter percorre a mesma ordenação do SelectKBest e salta
# uma coluna com |r| > 0.70 com outra já escolhida (W4, células 77 a 86). Com o mesmo k, a
# diferença emparelhada para o SelectKBest é o que vale só a regra da redundância (W4, passo 23).
# Com k=None não há limite: fica tudo o que não repete uma coluna já escolhida.
for k in K_VALUES + [None]:
    filter_runs[f"CorrelationFilter(k={k})"] = evaluate(
        prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL,
        select=CorrelationFilter(k, redundancy=REDUNDANT))

skip_rows = {}
for k in K_VALUES:
    pair = compare({f"k={k}": filter_runs[f"CorrelationFilter(k={k})"]},
                   filter_runs[f"SelectKBest(k={k})"])
    pair["2 x SEM"] = 2 * pair["SEM"]
    skip_rows[f"k={k}"] = pair.loc[f"k={k}", ["vs base", "SEM", "2 x SEM", "melhor em"]]
print("regra da redundância: CorrelationFilter contra SelectKBest com o mesmo k (F1)")
print(pd.DataFrame(skip_rows).T.round(4).to_string(), "\n")
print("colunas do CorrelationFilter(k=None), por divisão:",
      list(filter_runs["CorrelationFilter(k=None)"]["colunas"]))

regra da redundância: CorrelationFilter contra SelectKBest com o mesmo k (F1)
       vs base       SEM   2 x SEM melhor em
k=10  0.014889  0.001894  0.003788    5 de 5
k=20 -0.003781  0.004377  0.008754    1 de 5
k=40  0.003059  0.003643  0.007286    2 de 5
k=60 -0.003169  0.007495  0.014989    3 de 5
k=80  0.001651  0.004884  0.009768    3 de 5 

colunas do CorrelationFilter(k=None), por divisão: [66, 76, 88, 101, 113]


In [47]:
# Passo 46 - todos os filtros contra reference_3 (todas as colunas), nas mesmas 5 divisões.
# Só conta se "vs base" > 2 x SEM; um filtro que empata com menos colunas também é aceitável.
for metric in ["f1", "accuracy"]:
    print(f"[{metric}] contra todas as colunas (F1 {reference_3['f1'].mean():.4f},"
          f" accuracy {reference_3['accuracy'].mean():.4f})")
    print(filter_table(filter_runs, metric=metric).round(4).to_string(), "\n")

# as colunas que ficam na maioria das divisões, como a W4 mostra (só os filtros mais estreitos,
# para a lista ser legível)
for name in ["SelectKBest(k=10)", "SelectKBest(k=20)", "CorrelationFilter(k=10)"]:
    usual = kept_in_most(filter_runs[name])
    print(f"{name}: {len(usual)} colunas em 3 ou mais divisões")
    print("   ", ", ".join(usual), "\n")

# no k=None ficam quase todas: mais legível ver as que ele retira por repetirem outra. O
# VarianceThreshold(0) não removeu nada, por isso o seu `kept` é a lista de todas as colunas.
dropped = Counter(column
                  for every, kept in zip(filter_runs["VarianceThreshold(0)"]["kept"],
                                         filter_runs["CorrelationFilter(k=None)"]["kept"])
                  for column in set(every) - set(kept))
print("CorrelationFilter(k=None) retira, por repetirem uma coluna já escolhida:")
print("   ", ", ".join(f"{c} {n}/{len(splits)}" for c, n in dropped.most_common()
                       if n > len(splits) / 2))

[f1] contra todas as colunas (F1 0.7481, accuracy 0.7158)
                           colunas        f1  accuracy   vs base       SEM melhor em   2 x SEM
VarianceThreshold(0)         109.2  0.748076   0.71577       0.0       0.0    0 de 5       0.0
SelectKBest(k=10)             10.0  0.740249  0.704638 -0.007827  0.001987    0 de 5  0.003975
SelectKBest(k=20)             20.0  0.753311  0.718924  0.005235  0.004309    4 de 5  0.008619
SelectKBest(k=40)             40.0  0.746214  0.715399 -0.001862  0.004052    2 de 5  0.008104
SelectKBest(k=60)             60.0  0.747887  0.718367 -0.000189  0.004401    3 de 5  0.008802
SelectKBest(k=80)             80.0  0.744004  0.713915 -0.004072  0.002225    1 de 5   0.00445
CorrelationFilter(k=10)       10.0  0.755139    0.7141  0.007062  0.002193    5 de 5  0.004385
CorrelationFilter(k=20)       20.0   0.74953  0.714842  0.001454  0.002807    4 de 5  0.005615
CorrelationFilter(k=40)       40.0  0.749273  0.716141  0.001197  0.001952    3 de 5  0

In [48]:
# ===== [3.1] FILTROS - fim =====
# Passo 47 - o único filtro acima do ruído no F1 é o CorrelationFilter(k=10). Antes de o ler
# como melhor, ver porquê: o F1 sobe quando o modelo diz "1" mais vezes (passo 6: prever sempre
# 1 já dá F1 0.72), e a accuracy deste filtro não sobe. % de compras previstas como 1:
def share_of_ones(select):
    """% de previsões 1 em cada divisão, com a preparação final e o seletor `select`."""
    return [fit_predict(prepare_final, frame.iloc[t], frame.iloc[v], SCALER_FINAL,
                        ENCODER_FINAL, None, select).mean() for t, v in splits]


ones = pd.DataFrame({"todas as colunas": share_of_ones(None),
                     "CorrelationFilter(k=10)": share_of_ones(CorrelationFilter(10, REDUNDANT)),
                     "taxa real do alvo": [frame[TARGET].iloc[v].mean() for _, v in splits]})
print((100 * ones).round(1).to_string(), "\n\nmédia:", (100 * ones.mean()).round(1).to_dict())

# O que o 3.1 entrega ao 3.5 (números contra reference_3, F1 0.7481 e accuracy 0.7158):
#   - VarianceThreshold(0) não remove nenhuma coluna: não há colunas constantes (como na W4)
#   - SelectKBest: só o k=10 sai do ruído, e para pior (F1 -0.0078, 2 x SEM = 0.0040, 0 de 5);
#     k=20 a 80 empatam com todas as colunas; nenhum melhora a accuracy
#   - CorrelationFilter:
#       k=10   F1 +0.0071 (2 x SEM = 0.0044, melhor em 5 de 5), accuracy -0.0017 (ruído)
#       k=20 a 80 e k=None empatam nas duas métricas
#     o k=None tira 19 a 22 colunas (as cópias dos totais da fatura, das janelas de gasto e do
#     histórico) e empata: as cópias não estavam a ajudar
#   - a regra da redundância vale muito com k=10 (+0.0149 contra o SelectKBest(k=10), 5 de 5):
#     o SelectKBest gasta as 10 vagas em contagens e gastos que repetem o mesmo facto. Com k
#     maior já não faz diferença (ruído)
#   - cuidados para o 3.5 com o CorrelationFilter(k=10):
#       1. prevê 1 mais vezes do que a referência (em cima) e a accuracy não melhora: parte do
#          ganho no F1 é dizer "volta a comprar" mais vezes, não acertar mais
#       2. o k=10 foi escolhido entre 5 valores nas mesmas divisões que o medem, por isso o
#          ganho é otimista (W4, célula 100)
#       3. as colunas mudam de divisão para divisão: só 8 ficam em 3 ou mais das 5
print("\n3.1: tabela dos filtros no passo 46; a escolha do conjunto final é do 3.5")

   todas as colunas  CorrelationFilter(k=10)  taxa real do alvo
0              69.3                     64.4               58.4
1              55.4                     58.1               53.4
2              53.2                     62.0               57.6
3              50.4                     56.7               52.0
4              56.4                     63.0               60.4 

média: {'todas as colunas': 56.9, 'CorrelationFilter(k=10)': 60.8, 'taxa real do alvo': 56.4}

3.1: tabela dos filtros no passo 46; a escolha do conjunto final é do 3.5


In [49]:
# ===== [3.4] VERIFICAÇÕES DIRIGIDAS - início =====
# Passo 48 - as colunas que a tarefa 2 e a EDA deixaram em dúvida, e as famílias de cópias da
# EDA (passo 18: 23 pares acima de 0.70). Em cada família fica uma coluna, escolhida pelo
# significado e não pelos dados (escolher pela ligação ao alvo no treino inteiro seria olhar
# para o conjunto inteiro):
#   - fatura: Total a Pagar, o total final de que os outros são partes (Total Desc_ Linha fica:
#     é um desconto, não um total)
#   - contagens e gasto: a janela de 90 dias, a mesma do alvo
#   - histórico: as médias; o desvio só descreve a variação à volta da média
families = {
    "fatura: só Total a Pagar": ["Total Liquido", "Total Bruto", "Total Imp_", "Custo"],
    "contagens: só purchase_count_90d": ["purchase_count_30d", "purchase_count_180d",
                                         "purchase_count_365d", "prior_purchase_count"],
    "gasto: só spend_90d": ["spend_180d"],
    "histórico: só as médias": ["prior_std_purchase_value", "prior_std_gap_days"],
}
checks = {
    "sem random_noise": ["random_noise"],
    "sem transport_method": ["transport_method"],
    "sem commercial_zone": ["commercial_zone"],
    **families,
    "as 4 famílias juntas": [c for columns in families.values() for c in columns],
}


# prepare_without(columns) (preprocessing.py): prepare_final e depois retira `columns`, antes do
# one-hot e do modelo. Retirar depois da preparação dá o mesmo que retirar antes (a mediana, a
# moda e o log1p de cada coluna não dependem das outras) e mantém a lista do log1p e a regra de
# valores impossíveis do 2.1, que usa prior_max_purchase_value, tal como estavam.
from preprocessing import prepare_without

numeric, categorical = model_columns(frame)
pd.DataFrame({"retira": [", ".join(columns) for columns in checks.values()],
              "colunas antes do one-hot": [len(columns) for columns in checks.values()],
              "todas existem": [set(columns) <= set(numeric + categorical)
                                for columns in checks.values()]},
             index=list(checks))

,retira,colunas antes do one-hot,todas existem
sem random_noise,random_noise,1,True
sem transport_method,transport_method,1,True
sem commercial_zone,commercial_zone,1,True
fatura: só Total a Pagar,"Total Liquido, Total Bruto, Total Imp_, Custo",4,True
contagens: só purchase_count_90d,"purchase_count_30d, purchase_count_180d, purch...",4,True
gasto: só spend_90d,spend_180d,1,True
histórico: só as médias,"prior_std_purchase_value, prior_std_gap_days",2,True
as 4 famílias juntas,"Total Liquido, Total Bruto, Total Imp_, Custo,...",11,True


In [50]:
# Passo 49 - cada verificação contra reference_3 (todas as colunas), nas mesmas 5 divisões.
# "colunas" é o número médio que o modelo vê depois do one-hot (todas: 109.2).
# verdict(table) (preprocessing.py): "melhor" / "pior" se a diferença passar os 2 x SEM,
# senão "empata" (a regra da tarefa 2)
from preprocessing import verdict


def width(prepare):
    """Colunas que o modelo vê depois do one-hot, em média nas 5 divisões."""
    total = []
    for train_idx, val_idx in splits:
        train, _ = prepare(frame.iloc[train_idx], frame.iloc[val_idx])
        numeric, categorical = model_columns(train)
        total.append(len(numeric) + sum(train[c].nunique() for c in categorical))
    return np.mean(total)


check_runs = {name: evaluate(prepare_without(columns), frame, splits, scaler=SCALER_FINAL,
                             encoder=ENCODER_FINAL)
              for name, columns in checks.items()}
check_tables = {}
for metric in ["f1", "accuracy"]:
    table = compare(check_runs, reference_3, metric=metric)
    table["2 x SEM"] = 2 * table["SEM"]
    table["veredicto"] = verdict(table)
    if metric == "f1":
        table.insert(0, "colunas", [width(prepare_without(c)) for c in checks.values()])
    check_tables[metric] = table
    print(f"[{metric}] contra todas as colunas (F1 {reference_3['f1'].mean():.4f},"
          f" accuracy {reference_3['accuracy'].mean():.4f})")
    shown = [c for c in ["colunas", metric, "vs base", "SEM", "2 x SEM", "melhor em", "veredicto"]
             if c in table]
    print(table[shown].round(4).to_string(), "\n")

[f1] contra todas as colunas (F1 0.7481, accuracy 0.7158)
                                  colunas        f1   vs base       SEM   2 x SEM melhor em veredicto
sem random_noise                    108.2  0.749463  0.001387  0.001143  0.002287    3 de 5    empata
sem transport_method                 48.8  0.752546   0.00447  0.003436  0.006872    4 de 5    empata
sem commercial_zone                 105.8  0.748792  0.000716  0.000928  0.001855    2 de 5    empata
fatura: só Total a Pagar            105.2  0.747326  -0.00075   0.00116   0.00232    2 de 5    empata
contagens: só purchase_count_90d    105.2  0.751496   0.00342  0.001999  0.003998    4 de 5    empata
gasto: só spend_90d                 108.2  0.749973  0.001897  0.000963  0.001925    4 de 5    empata
histórico: só as médias             107.2  0.748114  0.000037  0.001595  0.003191    4 de 5    empata
as 4 famílias juntas                 98.2  0.752616   0.00454   0.00214  0.004281    5 de 5    melhor 

[accuracy] contra toda

In [51]:
# Passo 50 - uma decisão por linha. Regra da tarefa 3: uma remoção que empata sai, porque o
# modelo fica com menos colunas (W4, célula 89); se for pior, a coluna fica. Decide o F1
# (METRIC); por cautela, uma remoção que piore a accuracy acima do ruído também não sai.
decisions_34 = pd.DataFrame({
    "F1": check_tables["f1"]["veredicto"],
    "accuracy": check_tables["accuracy"]["veredicto"],
})
decisions_34["decisão"] = np.where(
    (decisions_34["F1"] != "pior") & (decisions_34["accuracy"] != "pior"), "retirar", "manter")
print(decisions_34.to_string(), "\n")

# tudo o que sai, junto: escolhas boas sozinhas podem não se somar (2.7). A linha "as 4 famílias
# juntas" já está incluída nas famílias, por isso não conta a dobrar.
to_drop = sorted({c for name, columns in checks.items()
                  if name != "as 4 famílias juntas" and decisions_34.loc[name, "decisão"] == "retirar"
                  for c in columns})
check_runs["tudo o que sai, junto"] = evaluate(prepare_without(to_drop), frame, splits,
                                               scaler=SCALER_FINAL, encoder=ENCODER_FINAL)
print(f"tudo o que sai ({len(to_drop)} colunas antes do one-hot): {', '.join(to_drop)}")
print(f"o modelo passa a ver {width(prepare_without(to_drop)):.1f} colunas em média (todas: 109.2)\n")
for metric in ["f1", "accuracy"]:
    joint = compare({"tudo o que sai, junto": check_runs["tudo o que sai, junto"]},
                    reference_3, metric=metric)
    joint["2 x SEM"] = 2 * joint["SEM"]
    joint["veredicto"] = verdict(joint)
    print(f"[{metric}]", joint[[metric, "vs base", "SEM", "2 x SEM", "melhor em", "veredicto"]]
          .round(4).to_string(header=metric == "f1"))

                                      F1 accuracy  decisão
sem random_noise                  empata   empata  retirar
sem transport_method              empata   melhor  retirar
sem commercial_zone               empata   empata  retirar
fatura: só Total a Pagar          empata   empata  retirar
contagens: só purchase_count_90d  empata   empata  retirar
gasto: só spend_90d               empata   empata  retirar
histórico: só as médias           empata   empata  retirar
as 4 famílias juntas              melhor   empata  retirar 



tudo o que sai (14 colunas antes do one-hot): Custo, Total Bruto, Total Imp_, Total Liquido, commercial_zone, prior_purchase_count, prior_std_gap_days, prior_std_purchase_value, purchase_count_180d, purchase_count_30d, purchase_count_365d, random_noise, spend_180d, transport_method


o modelo passa a ver 33.4 colunas em média (todas: 109.2)

[f1]                              f1   vs base       SEM   2 x SEM melhor em veredicto
tudo o que sai, junto  0.750832  0.002756  0.002351  0.004701    3 de 5    empata
[accuracy] tudo o que sai, junto  0.716327  0.000557  0.00265  0.0053  2 de 5  empata


In [52]:
# ===== [3.4] VERIFICAÇÕES DIRIGIDAS - fim =====
# Passo 51 - o que o 3.4 entrega ao 3.5 (números contra reference_3, F1 0.7481, accuracy 0.7158):
#   - todas as remoções empatam no F1 e nenhuma piora a accuracy, por isso todas saem:
#       random_noise       F1 +0.0014, accuracy +0.0009: não mede nada, como a EDA viu
#       transport_method   F1 +0.0045 (ruído), accuracy +0.0085 (2 x SEM = 0.0082, melhor):
#                          sai e leva consigo 60 das 109 colunas do one-hot
#       commercial_zone    F1 +0.0007, accuracy +0.0011: quase não se liga ao alvo (V 0.06)
#       as 4 famílias      cada uma sozinha empata; as quatro juntas melhoram o F1 (+0.0045,
#                          2 x SEM = 0.0043, 5 de 5) e a accuracy empata
#   - tudo junto (14 colunas antes do one-hot): o modelo passa de 109 para 33 colunas e empata
#     nas duas métricas (F1 +0.0028, accuracy +0.0006): o mesmo resultado com 3 vezes menos
#     colunas
#   - cuidado: as colunas que ficam em cada família foram escolhidas pelo significado, antes de
#     medir; o "tudo junto" só foi medido depois de ver as linhas sozinhas, por isso é uma
#     confirmação, não uma escolha otimizada
from preprocessing import DROP_34
assert DROP_34 == to_drop   # preprocessing.py tem exatamente as colunas que o 3.4 decidiu
print(f"3.4: {len(DROP_34)} colunas a retirar, para o 3.5 decidir: {', '.join(DROP_34)}")

3.4: 14 colunas a retirar, para o 3.5 decidir: Custo, Total Bruto, Total Imp_, Total Liquido, commercial_zone, prior_purchase_count, prior_std_gap_days, prior_std_purchase_value, purchase_count_180d, purchase_count_30d, purchase_count_365d, random_noise, spend_180d, transport_method


In [53]:
# ===== [3.2] WRAPPER (RFE) - início =====
# Passo 52 - RFE: um wrapper julga as colunas através de um modelo treinado (W4, "Feature
# selection II", células 90 a 100). Treina a regressão logística, tira a coluna com o peso mais
# pequeno (em valor absoluto), volta a treinar, e repete até restarem k colunas.
# - só o RFE: o RFECV e o SequentialFeatureSelector fazem validação interna, e com a preparação
#   feita fora deles haveria fuga de informação (W4, célula 93)
# - o k não se sabe à partida: mede-se 5, 10, 15, ... até à divisão mais estreita (85 colunas),
#   e fica o k com melhor F1 médio; num empate exato, o mais pequeno (W4, célula 95)
# - o RFE é ajustado só no treino de cada divisão (select= do 3.0)
from sklearn.feature_selection import RFE   # aqui, como os imports do 3.1

rfe_widths = list(range(5, min(widths) + 1, 5))
rfe_runs = {k: evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL,
                        select=RFE(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                                   n_features_to_select=k))
            for k in rfe_widths}

rfe_curve = pd.DataFrame({
    "k": rfe_widths,
    "F1 médio": [rfe_runs[k]["f1"].mean() for k in rfe_widths],
    "F1 SEM": [sem(rfe_runs[k]["f1"]) for k in rfe_widths],
    "accuracy média": [rfe_runs[k]["accuracy"].mean() for k in rfe_widths],
})
best_f1 = rfe_curve["F1 médio"].max()
chosen_k = int(rfe_curve.loc[rfe_curve["F1 médio"] == best_f1, "k"].min())
rfe_curve["escolhido"] = rfe_curve["k"] == chosen_k
print(rfe_curve.round(4).to_string(index=False))
print(f"\nk escolhido = {chosen_k}, pelo F1 médio; num empate exato fica o mais pequeno")
print(f"referência (todas as colunas, 85 a 133): F1 {reference_3['f1'].mean():.4f}")

 k  F1 médio  F1 SEM  accuracy média  escolhido
 5    0.7455  0.0092          0.7078      False
10    0.7490  0.0099          0.7106      False
15    0.7534  0.0092          0.7139      False
20    0.7532  0.0080          0.7152      False
25    0.7546  0.0084          0.7174       True
30    0.7530  0.0088          0.7167      False
35    0.7505  0.0085          0.7167      False
40    0.7503  0.0082          0.7191      False
45    0.7516  0.0092          0.7226      False
50    0.7477  0.0094          0.7173      False
55    0.7465  0.0110          0.7180      False
60    0.7441  0.0108          0.7150      False
65    0.7427  0.0105          0.7147      False
70    0.7431  0.0112          0.7141      False
75    0.7438  0.0110          0.7141      False
80    0.7419  0.0115          0.7122      False
85    0.7424  0.0095          0.7115      False

k escolhido = 25, pelo F1 médio; num empate exato fica o mais pequeno
referência (todas as colunas, 85 a 133): F1 0.7481


In [54]:
# Passo 53 - o k escolhido contra todas as colunas (reference_3), nas duas métricas, e as
# mesmas verificações do 3.1: quantas vezes prevê 1 e que colunas ficam na maioria das divisões
rfe_best = {f"RFE(k={chosen_k})": rfe_runs[chosen_k]}
for metric in ["f1", "accuracy"]:
    table = compare(rfe_best, reference_3, metric=metric)
    table["2 x SEM"] = 2 * table["SEM"]
    table["veredicto"] = verdict(table)
    print(f"[{metric}]", table[[metric, "vs base", "SEM", "2 x SEM", "melhor em", "veredicto"]]
          .round(4).to_string(header=metric == "f1"))

ones_rfe = pd.DataFrame({
    "todas as colunas": share_of_ones(None),
    f"RFE(k={chosen_k})": share_of_ones(RFE(LogisticRegression(max_iter=1000,
                                                                random_state=RANDOM_STATE),
                                            n_features_to_select=chosen_k)),
    "taxa real do alvo": [frame[TARGET].iloc[v].mean() for _, v in splits],
})
print("\n% de compras previstas como 1, por divisão:")
print((100 * ones_rfe).round(1).to_string())

usual = kept_in_most(rfe_runs[chosen_k])
print(f"\nRFE(k={chosen_k}): {len(usual)} colunas em 3 ou mais divisões")
print("   ", ", ".join(usual))

[f1]                  f1   vs base       SEM   2 x SEM melhor em veredicto
RFE(k=25)  0.754581  0.006505  0.001414  0.002829    5 de 5    melhor
[accuracy] RFE(k=25)  0.71744  0.00167  0.004375  0.008749  3 de 5  empata



% de compras previstas como 1, por divisão:
   todas as colunas  RFE(k=25)  taxa real do alvo
0              69.3       66.9               58.4
1              55.4       60.1               53.4
2              53.2       55.8               57.6
3              50.4       53.0               52.0
4              56.4       60.1               60.4

RFE(k=25): 23 colunas em 3 ou mais divisões
    Total Imp_ 5/5, Total Desc_ Linha 5/5, prior_mean_purchase_value 5/5, prior_max_purchase_value 5/5, prior_std_gap_days 5/5, purchase_count_90d 5/5, purchase_count_365d 5/5, salesperson_VLOOKUP Master 8 5/5, transport_method_Maybe Tomorrow Delivery 11 5/5, transport_method_Maybe Tomorrow Delivery 18 5/5, transport_method_Maybe Tomorrow Delivery 68 5/5, transport_method_Maybe Tomorrow Delivery 39 4/5, transport_method_Maybe Tomorrow Delivery 50 4/5, transport_method_Maybe Tomorrow Delivery 80 4/5, transport_method_Maybe Tomorrow Delivery 9 4/5, prior_mean_gap_days 4/5, transport_method_Maybe Tomorrow 

In [55]:
# ===== [3.2] WRAPPER (RFE) - fim =====
# Passo 54 - o que o 3.2 entrega ao 3.5 (números contra reference_3, F1 0.7481, accuracy 0.7158):
#   - RFE(k=25): F1 0.7546, +0.0065 com 2 x SEM = 0.0028, melhor em 5 de 5 divisões;
#     accuracy +0.0017, dentro do ruído (2 x SEM = 0.0087)
#   - a curva é plana: de k=15 a k=45 o F1 fica entre 0.750 e 0.755, perto uns dos outros. O
#     k=25 é a melhor média, mas foi escolhido nas mesmas divisões que o medem, por isso o
#     ganho é otimista (W4, célula 100)
#   - prevê 1 em 59% das compras, contra 57% com todas as colunas e 56% na realidade (passo 53):
#     como no CorrelationFilter(k=10) do 3.1 (61%), parte do ganho no F1 vem de dizer
#     "volta a comprar" mais vezes; a accuracy não sobe
#   - que colunas ficam: 12 das 23 mais frequentes são níveis da transport_method, e um nível
#     do salesperson com 3 compras em todo o treino (VLOOKUP Master 8) fica nas 5 divisões.
#     O RFE tira primeiro as colunas com o peso mais pequeno na regressão logística, e um
#     nível raro com uma taxa extrema (Delivery 11: 98% de recompra em 253 compras; Master 8:
#     3 compras, todas com recompra) recebe um peso grande. É um risco de decorar níveis raros,
#     e vai no sentido contrário do 3.4, onde retirar a transport_method empatava
#   - para o 3.5: o RFE(k=25) é candidato no F1; vale a pena medi-lo também depois de retirar
#     as colunas do 3.4 (DROP_34), para ver se o ganho vem dos níveis raros
RFE_K = chosen_k
print(f"3.2: RFE com k={RFE_K} entregue ao 3.5 (F1 {rfe_runs[RFE_K]['f1'].mean():.4f})")

3.2: RFE com k=25 entregue ao 3.5 (F1 0.7546)


In [56]:
# ===== [3.3] EMBEDDED - início =====
# Passo 55 - métodos embedded: a seleção acontece dentro do próprio modelo (W4, "Feature
# selection III", células 103 a 112). O SelectFromModel treina o modelo e decide pelo `threshold`:
# - L1: a regressão logística com penalização L1 põe a zero os pesos das colunas fracas;
#   threshold=1e-8 fica com as colunas que a penalização deixou vivas (W4, célula 106).
#   l1_ratio=1 é a penalização L1 (lasso); C=0.1 fixa a força, como na W4
# - árvore: a importância de cada coluna numa árvore de decisão; threshold="median" fica com a
#   metade mais importante, uma largura escolhida à partida (W4, célula 110)
from sklearn.feature_selection import SelectFromModel   # aqui, como os imports do 3.1 e do 3.2
from sklearn.tree import DecisionTreeClassifier


def l1_selector():
    """O seletor L1 da W4, novo em cada chamada."""
    return SelectFromModel(LogisticRegression(l1_ratio=1, solver="liblinear", C=0.1,
                                              max_iter=1000, random_state=RANDOM_STATE),
                           threshold=1e-8)


def tree_selector():
    """O seletor por importância da árvore da W4, novo em cada chamada."""
    return SelectFromModel(DecisionTreeClassifier(random_state=RANDOM_STATE), threshold="median")


embedded_runs = {"L1 (C=0.1)": evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL,
                                        encoder=ENCODER_FINAL, select=l1_selector()),
                 "árvore (median)": evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL,
                                             encoder=ENCODER_FINAL, select=tree_selector())}
for metric in ["f1", "accuracy"]:
    table = compare(embedded_runs, reference_3, metric=metric)
    table["2 x SEM"] = 2 * table["SEM"]
    table["veredicto"] = verdict(table)
    if metric == "f1":
        table.insert(0, "colunas", [embedded_runs[n]["colunas"].mean() for n in table.index])
    shown = [c for c in ["colunas", metric, "vs base", "SEM", "2 x SEM", "melhor em", "veredicto"]
             if c in table]
    print(f"[{metric}] contra todas as colunas (F1 {reference_3['f1'].mean():.4f})")
    print(table[shown].round(4).to_string(), "\n")

[f1] contra todas as colunas (F1 0.7481)
                 colunas        f1   vs base       SEM   2 x SEM melhor em veredicto
L1 (C=0.1)          17.6  0.749786   0.00171  0.003742  0.007483    3 de 5    empata
árvore (median)     55.0  0.750247  0.002171  0.003453  0.006906    3 de 5    empata 

[accuracy] contra todas as colunas (F1 0.7481)
                 accuracy   vs base       SEM   2 x SEM melhor em veredicto
L1 (C=0.1)       0.714842 -0.000928  0.005756  0.011512    3 de 5    empata
árvore (median)  0.718738  0.002968   0.00253  0.005061    3 de 5    empata 



In [57]:
# Passo 56 - que colunas cada método mantém, e quantas vezes prevê 1 (como no 3.1 e no 3.2)
for name in embedded_runs:
    usual = kept_in_most(embedded_runs[name])
    print(f"{name}: {len(usual)} colunas em 3 ou mais divisões")
    print("   ", ", ".join(usual), "\n")

ones_33 = pd.DataFrame({
    "todas as colunas": share_of_ones(None),
    "L1 (C=0.1)": share_of_ones(l1_selector()),
    "árvore (median)": share_of_ones(tree_selector()),
    "taxa real do alvo": [frame[TARGET].iloc[v].mean() for _, v in splits],
})
print("% de compras previstas como 1, média das 5 divisões:")
print((100 * ones_33.mean()).round(1).to_string())

L1 (C=0.1): 20 colunas em 3 ou mais divisões
    Total Desc_ Linha 5/5, purchase_count_90d 5/5, spend_90d 5/5, purchase_count_365d 5/5, transport_method_Maybe Tomorrow Delivery 72 5/5, warehouse_Final Excel Warehouse 1 5/5, prior_max_purchase_value 4/5, prior_mean_gap_days 4/5, prior_std_gap_days 4/5, salesperson_VLOOKUP Master 4 4/5, transport_method_Maybe Tomorrow Delivery 68 4/5, document_series_Ctrl Z Series 3 3/5, transport_method_Maybe Tomorrow Delivery 53 3/5, Total Imp_ 3/5, N_ Detalhes 3/5, spend_30d 3/5, payment_type_Maybe Tomorrow Payment 7 3/5, payment_type_Maybe Tomorrow Payment 9 3/5, transport_method_Maybe Tomorrow Delivery 59 3/5, transport_method_Maybe Tomorrow Delivery 73 3/5 

árvore (median): 56 colunas em 3 ou mais divisões
    Total a Pagar 5/5, Total Bruto 5/5, Total Liquido 5/5, Total Imp_ 5/5, Total Portes 5/5, Total Desc_ Global 5/5, Total Desc_ Linha 5/5, Custo 5/5, N_ Detalhes 5/5, invoice_count_current_purchase 5/5, days_since_previous_purchase 5/5, custome

% de compras previstas como 1, média das 5 divisões:
todas as colunas     56.9
L1 (C=0.1)           58.0
árvore (median)      56.9
taxa real do alvo    56.4


In [58]:
# ===== [3.3] EMBEDDED - fim =====
# Passo 57 - o que o 3.3 entrega ao 3.5 (números contra reference_3, F1 0.7481):
#   - L1 (C=0.1): 17.6 colunas em média, F1 +0.0017 (2 x SEM = 0.0075), empata; accuracy
#     empata; prevê 1 em 58% das compras. Mantém só um sexto das colunas sem perder nada
#   - árvore (median): 55 colunas, F1 +0.0022 (2 x SEM = 0.0069), empata; accuracy empata.
#     Mantém o random_noise nas 5 divisões: uma árvore sem limite de profundidade divide em
#     qualquer coluna, até numa de puro ruído, e a importância que lhe dá não quer dizer que
#     ajude. As importâncias da árvore não servem para escolher colunas aqui (W4, célula 112:
#     "separates columns whose importances differ in the third decimal place")
#   - nenhum dos dois é melhor do que todas as colunas; o L1 entra na votação do 3.5
print("3.3: L1 e árvore empatam com todas as colunas; o L1 entra na votação do 3.5")

3.3: L1 e árvore empatam com todas as colunas; o L1 entra na votação do 3.5


In [59]:
# ===== [3.5] COMBINAÇÃO E CONJUNTO FINAL - início =====
# Passo 58 - votação (W4, "Combining strategies", células 115 a 130): três métodos, um de cada
# família, ajustados no mesmo treino; uma coluna fica se pelo menos 2 dos 3 a mantiverem. Na
# aula, a votação foi a combinação que não perdeu (a sequência, em funil, perdia: W4, célula 125).
# - filtro: CorrelationFilter(k=40) do 3.1 (o mais largo dos que empatam), wrapper: RFE(k=25)
#   do 3.2, embedded: L1 do 3.3
class Vote:
    """Fica uma coluna que pelo menos `minimum` dos `members` mantêm (W4, célula 118).

    Acrescentei `transform` e `get_feature_names_out`, que o fit_predict e o evaluate do 3.0
    usam (como no CorrelationFilter do 3.1); a votação é a da W4.
    """

    def __init__(self, members, minimum):
        self.members = members
        self.minimum = minimum

    def fit(self, frame, outcome):
        self.feature_names_in_ = frame.columns.to_numpy()
        self.decisions_ = pd.DataFrame(
            {name: member.fit(frame, outcome).get_support()
             for name, member in self.members.items()},
            index=frame.columns,
        )
        self.support_ = (self.decisions_.sum(axis=1) >= self.minimum).to_numpy()
        self.decisions_["kept"] = self.support_
        return self

    def get_support(self):
        return self.support_

    def transform(self, frame):
        return np.asarray(frame)[:, self.support_]

    def get_feature_names_out(self):
        return self.feature_names_in_[self.support_]


def vote_selector():
    """Os três métodos da votação, novos em cada chamada (W4, célula 120)."""
    return Vote({"filtro": CorrelationFilter(40, redundancy=REDUNDANT),
                 "wrapper": RFE(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                                n_features_to_select=RFE_K),
                 "embedded": l1_selector()},
                minimum=2)


vote_run = evaluate(prepare_final, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL,
                    select=vote_selector())
print(f"votação: {vote_run['colunas'].mean():.1f} colunas em média,"
      f" F1 {vote_run['f1'].mean():.4f}, accuracy {vote_run['accuracy'].mean():.4f}")
usual = kept_in_most(vote_run)
print(f"{len(usual)} colunas em 3 ou mais divisões:", ", ".join(usual))

votação: 24.2 colunas em média, F1 0.7484, accuracy 0.7158
25 colunas em 3 ou mais divisões: Total Desc_ Linha 5/5, N_ Detalhes 5/5, prior_max_purchase_value 5/5, prior_std_gap_days 5/5, purchase_count_90d 5/5, purchase_count_365d 5/5, transport_method_Maybe Tomorrow Delivery 11 5/5, transport_method_Maybe Tomorrow Delivery 68 5/5, transport_method_Maybe Tomorrow Delivery 72 5/5, transport_method_Maybe Tomorrow Delivery 39 4/5, transport_method_Maybe Tomorrow Delivery 80 4/5, transport_method_Maybe Tomorrow Delivery 9 4/5, prior_mean_gap_days 4/5, warehouse_Final Excel Warehouse 1 4/5, spend_90d 3/5, salesperson_VLOOKUP Master 8 3/5, transport_method_Maybe Tomorrow Delivery 50 3/5, transport_method_Maybe Tomorrow Delivery 53 3/5, Total Imp_ 3/5, payment_type_Maybe Tomorrow Payment 7 3/5, payment_type_Maybe Tomorrow Payment 9 3/5, transport_method_Maybe Tomorrow Delivery 22 3/5, transport_method_Maybe Tomorrow Delivery 48 3/5, transport_method_Maybe Tomorrow Delivery 59 3/5, transport_m

In [60]:
# Passo 59 - todos os candidatos lado a lado, contra todas as colunas (reference_3), nas mesmas
# 5 divisões. Inclui o RFE(k=25) depois de retirar as colunas do 3.4: se o ganho do RFE vier dos
# níveis raros da transport_method (passo 54), desaparece sem ela.
rfe_after_34 = evaluate(prepare_without(DROP_34), frame, splits, scaler=SCALER_FINAL,
                        encoder=ENCODER_FINAL,
                        select=RFE(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                                   n_features_to_select=RFE_K))
candidates = {
    "3.1 CorrelationFilter(k=10)": filter_runs["CorrelationFilter(k=10)"],
    "3.2 RFE(k=25)": rfe_runs[RFE_K],
    "3.3 L1": embedded_runs["L1 (C=0.1)"],
    "3.4 sem as 14 colunas": check_runs["tudo o que sai, junto"],
    "3.2 + 3.4 RFE(k=25) sem as 14": rfe_after_34,
    "3.5 votação 2 de 3": vote_run,
}
widths_of = {name: (run["colunas"].mean() if "colunas" in run else width(prepare_without(DROP_34)))
             for name, run in candidates.items()}


def ones_share(prepare, select=None):
    """% média de previsões 1 nas 5 divisões, com uma preparação e um seletor quaisquer."""
    return 100 * np.mean([fit_predict(prepare, frame.iloc[t], frame.iloc[v], SCALER_FINAL,
                                      ENCODER_FINAL, None, select).mean() for t, v in splits])


ones_of = {
    "3.1 CorrelationFilter(k=10)": ones_share(prepare_final, CorrelationFilter(10, REDUNDANT)),
    "3.2 RFE(k=25)": ones_share(prepare_final, RFE(LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE), n_features_to_select=RFE_K)),
    "3.3 L1": ones_share(prepare_final, l1_selector()),
    "3.4 sem as 14 colunas": ones_share(prepare_without(DROP_34)),
    "3.2 + 3.4 RFE(k=25) sem as 14": ones_share(prepare_without(DROP_34), RFE(LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE), n_features_to_select=RFE_K)),
    "3.5 votação 2 de 3": ones_share(prepare_final, vote_selector()),
}

f1_table = compare(candidates, reference_3, metric="f1")
acc_table = compare(candidates, reference_3, metric="accuracy")
summary = pd.DataFrame({
    "colunas": pd.Series(widths_of),
    "F1": f1_table["f1"],
    "F1 vs todas": f1_table["vs base"],
    "F1 2 x SEM": 2 * f1_table["SEM"],
    "F1 melhor em": f1_table["melhor em"],
    "F1 veredicto": verdict(f1_table),
    "accuracy vs todas": acc_table["vs base"],
    "accuracy veredicto": verdict(acc_table),
    "% previsões 1": pd.Series(ones_of),
})
print(f"todas as colunas: F1 {reference_3['f1'].mean():.4f}, accuracy"
      f" {reference_3['accuracy'].mean():.4f}, colunas {reference_3.shape and 109.2},"
      f" % previsões 1 {100 * ones_33['todas as colunas'].mean():.1f}"
      f" (taxa real {100 * ones_33['taxa real do alvo'].mean():.1f})\n")
print(summary.round(4).to_string())

todas as colunas: F1 0.7481, accuracy 0.7158, colunas 109.2, % previsões 1 56.9 (taxa real 56.4)

                               colunas        F1 F1 vs todas F1 2 x SEM F1 melhor em F1 veredicto accuracy vs todas accuracy veredicto  % previsões 1
3.1 CorrelationFilter(k=10)       10.0  0.755139    0.007062   0.004385       5 de 5       melhor          -0.00167             empata        60.8163
3.2 RFE(k=25)                     25.0  0.754581    0.006505   0.002829       5 de 5       melhor           0.00167             empata        59.1837
3.3 L1                            17.6  0.749786     0.00171   0.007483       3 de 5       empata         -0.000928             empata        57.9963
3.4 sem as 14 colunas             33.4  0.750832    0.002756   0.004701       3 de 5       empata          0.000557             empata        57.8850
3.2 + 3.4 RFE(k=25) sem as 14     25.0  0.749875    0.001799   0.007448       2 de 5       empata               0.0             empata        57.6438
3.

In [61]:
# Passo 60 - o conjunto final de features (escolha da equipa, a partir da tabela do passo 59):
# retirar as 14 colunas do 3.4, antes do one-hot. Porquê esta e não as que ganham no F1:
#   1. empata com todas as colunas nas duas métricas (F1 +0.0028, accuracy +0.0006) com 3 vezes
#      menos colunas (33 em vez de 109)
#   2. tira o que já se sabia ser um problema: o random_noise (não mede nada), a transport_method
#      (80 níveis, 51 com menos de 30 compras), a commercial_zone (21% do teste em zonas que o
#      treino nunca viu, passo 36) e as cópias das famílias redundantes (EDA, passo 18)
#   3. as colunas foram escolhidas pelo significado, antes de medir: o resultado não é otimista
#   4. o RFE(k=25) e o CorrelationFilter(k=10) ganham no F1, mas o k foi escolhido nas mesmas
#      divisões que os medem, parte do ganho vem de preverem 1 mais vezes (59% e 61%, contra
#      56% na realidade) e o ganho do RFE desaparece sem a transport_method (+0.0018, ruído):
#      vinha dos níveis raros
# Em preprocessing.py: prepare_selected = prepare_without(DROP_34) (prepare_final + retirar as
# 14 colunas) e SELECT_FINAL = None (nenhum seletor por cima: as colunas já estão escolhidas)
from preprocessing import SELECT_FINAL, prepare_selected
selected_run = check_runs["tudo o que sai, junto"]

selection_decisions = pd.DataFrame([
    ("3.0", "identificadores (ID, customer_id, purchase_date)", "-", "ficam de fora: não descrevem a compra"),
    ("3.1", "VarianceThreshold, SelectKBest, CorrelationFilter", "CorrelationFilter(k=10) +0.0071",
     "ganho otimista e vindo de prever mais 1s; não fica"),
    ("3.2", "RFE com k de 5 a 85", "RFE(k=25) +0.0065",
     "depende de níveis raros da transport_method; sem eles, ruído; não fica"),
    ("3.3", "L1 e importância da árvore", "empatam", "a árvore mantém o random_noise; não ficam"),
    ("3.4", "retirar ruído, colunas problemáticas e cópias", "empata com 33 colunas",
     "fica: o mesmo resultado, 3 vezes menos colunas, escolhidas pelo significado"),
    ("3.5", "votação 2 de 3 (filtro, RFE, L1)", "empata", "não acrescenta nada ao 3.4; não fica"),
], columns=["pacote", "método", "resultado no F1", "decisão"])
print(selection_decisions.to_string(index=False))
print(f"\nconjunto final: {width(prepare_selected):.1f} colunas, F1 {selected_run['f1'].mean():.4f}"
      f" +- {sem(selected_run['f1']):.4f}, accuracy {selected_run['accuracy'].mean():.4f}"
      f" (todas as colunas: F1 {reference_3['f1'].mean():.4f})")
print("retiradas:", ", ".join(DROP_34))

pacote                                            método                 resultado no F1                                                                     decisão
   3.0  identificadores (ID, customer_id, purchase_date)                               -                                       ficam de fora: não descrevem a compra
   3.1 VarianceThreshold, SelectKBest, CorrelationFilter CorrelationFilter(k=10) +0.0071                          ganho otimista e vindo de prever mais 1s; não fica
   3.2                               RFE com k de 5 a 85               RFE(k=25) +0.0065      depende de níveis raros da transport_method; sem eles, ruído; não fica
   3.3                        L1 e importância da árvore                         empatam                                   a árvore mantém o random_noise; não ficam
   3.4     retirar ruído, colunas problemáticas e cópias           empata com 33 colunas fica: o mesmo resultado, 3 vezes menos colunas, escolhidas pelo significado
   3.5    


conjunto final: 33.4 colunas, F1 0.7508 +- 0.0074, accuracy 0.7163 (todas as colunas: F1 0.7481)
retiradas: Custo, Total Bruto, Total Imp_, Total Liquido, commercial_zone, prior_purchase_count, prior_std_gap_days, prior_std_purchase_value, purchase_count_180d, purchase_count_30d, purchase_count_365d, random_noise, spend_180d, transport_method


In [62]:
# Passo 61 - o conjunto final aplicado ao test.csv: as mesmas verificações do 2.7 (passo 36)
train_selected, test_selected = prepare_selected(frame, test_csv)
numeric, categorical = model_columns(train_selected)
pd.Series({
    "colunas antes do one-hot: numéricas / categóricas": f"{len(numeric)} / {len(categorical)}",
    "mesmas colunas no treino e no teste": list(model_columns(test_selected)[0]) == numeric
                                           and list(model_columns(test_selected)[1]) == categorical,
    "nulos no treino / teste": f"{train_selected[numeric + categorical].isna().sum().sum()}"
                               f" / {test_selected[numeric + categorical].isna().sum().sum()}",
    "linhas do teste com um nível novo, por coluna":
        {c: int((~test_selected[c].isin(train_selected[c])).sum()) for c in categorical},
}).to_frame("verificação")

,verificação
colunas antes do one-hot: numéricas / categóricas,17 / 4
mesmas colunas no treino e no teste,True
nulos no treino / teste,0 / 0
"linhas do teste com um nível novo, por coluna","{'payment_type': 3, 'salesperson': 0, 'warehou..."


In [63]:
# ===== [3.5] COMBINAÇÃO E CONJUNTO FINAL - fim =====
# Passo 62 - o que a tarefa 4 usa daqui em diante:
#   - prepare_selected (a preparação final do 2.7 sem as 14 colunas), ENCODER_FINAL e
#     SCALER_FINAL, com select=None
#   - evaluate(prepare_selected, frame, splits, scaler=SCALER_FINAL, encoder=ENCODER_FINAL,
#              model=...) para comparar modelos; selected_run é a referência da tarefa 4
print(f"referência da tarefa 4: F1 {selected_run['f1'].mean():.4f}"
      f" +- {sem(selected_run['f1']):.4f}, accuracy {selected_run['accuracy'].mean():.4f}")

referência da tarefa 4: F1 0.7508 +- 0.0074, accuracy 0.7163


In [64]:
# ===== SUBMISSÃO AO KAGGLE =====
# A submissão passou para o fim do 03_modelling.ipynb, onde se escolhe o modelo (tarefa 4).
# Fica aqui em comentário, como pede o enunciado do projeto para o código que ajudou a
# decidir: era esta célula que gerava a Version03 e a Version04.
# # Passo final - a preparação final (2.7) com as features escolhidas na tarefa 3 (sem as 14
# # colunas do 3.4) e o modelo, ajustados no treino inteiro, preveem as 986 compras do test.csv.
# # Formato do sample_submission: ID e previsão (0 ou 1).
# # VERSION é escrito à mão: correr o notebook outra vez volta a gerar o mesmo ficheiro, e o
# # notebook entregue tem de reproduzir a submissão escolhida no Kaggle.
# TEAM = "ML_Group_37"
# VERSION = 4   # Version03 = preparação do 2.7; Version04 = + features da tarefa 3
#
# predictions = fit_predict(prepare_selected, frame, test_csv, scaler=SCALER_FINAL,
#                           encoder=ENCODER_FINAL, select=SELECT_FINAL)
# submission = pd.DataFrame({"ID": test_csv.index, TARGET: predictions})
#
# path = f"../submissions/{TEAM}_Version{VERSION:02d}.csv"
# submission.to_csv(path, index=False)   # sem o índice do pandas, como pede o guia do Kaggle
#
# sample = pd.read_csv("../data/sample_submission.csv")
# print(f"gravado em {path}")
# print(f"linhas: {len(submission)}, IDs únicos: {submission['ID'].nunique()},"
#       f" mesmos IDs do sample_submission: {submission['ID'].isin(sample['ID']).all()}")
# print("previsões:", submission[TARGET].value_counts().sort_index().to_dict())